# Run the pipeline on all four C-MAPSS subsets

Runs the full pipeline from `pipeline_FD001.ipynb` (ingest → features → CV model selection → retraining and promotion → evaluation → triage agent → export) on FD001, FD002, FD003 and FD004, one after another, without manual steps.

- Each subset runs in a **fresh namespace**, the same as restarting the kernel and changing `SUBSET` by hand.
- If one subset fails, its error is recorded and the **next subset still runs**.
- `CLEAN_START = True` wipes earlier artifacts and the tracking store first, so outputs only contain this run.
- At the end: a **cross-subset summary table**, the **monitoring dashboard** (`dashboard.html`, one offline page, inside the zip) and **one zip** with every subset's artifacts plus one MLflow store (one experiment per subset): `/kaggle/working/all_subsets_outputs.zip`.

**Run on Kaggle**: attach the dataset, *Accelerator → GPU T4 x2*, *Internet → On*, then *Run all*. To run only some subsets, edit `SUBSETS`. The pipeline code is embedded below and generated from the same source as `pipeline_FD001.ipynb` (`scripts/build_notebook.py`); edit it there, not here.

In [ ]:
# MLflow is not preinstalled on Kaggle. If the install fails, the pipeline falls back to JSON tracking.
import importlib.util, subprocess, sys

def ensure(module, pip_name=None):
    if importlib.util.find_spec(module) is None:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module],
                           capture_output=True, text=True)
        print(f"pip install {pip_name or module}: {'ok' if r.returncode == 0 else 'FAILED'}")

ensure("mlflow")

In [ ]:
import json

# pipeline code cells, generated from scripts/build_notebook.py
PIPELINE_CELLS = json.loads(r'''[
"import json, shutil, subprocess, time, warnings\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, IsolationForest\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nwarnings.filterwarnings(\"ignore\", category=UserWarning)\nwarnings.filterwarnings(\"ignore\", category=FutureWarning)\ntry:\n    import xgboost as xgb\nexcept ImportError:\n    xgb = None\n\nSUBSET = \"FD001\"  # FD001 / FD002 / FD003 / FD004 (run_all_subsets.ipynb rewrites this line)\nSEED = 42\nRUN_TAG = \"\"  # run_all_subsets sets e.g. \"_seed1\" for extra seeds, so their outputs don't overwrite seed 42's\n\nSUBSET_CONFIG = {\n    \"FD001\": {\"condition_norm\": False, \"fault_modes\": 1},\n    \"FD002\": {\"condition_norm\": True, \"fault_modes\": 1},\n    \"FD003\": {\"condition_norm\": False, \"fault_modes\": 2},\n    \"FD004\": {\"condition_norm\": True, \"fault_modes\": 2},\n}\n\nCFG = {\n    \"subset\": SUBSET, **SUBSET_CONFIG[SUBSET],\n    # labels\n    \"rul_cap\": 125,             # piecewise-linear target: flat while the engine is healthy\n    \"fail_window\": 30,          # \"failing\" class for the early-warning classifier: RUL <= 30\n    # cleaning / features\n    \"min_unique\": 2,            # drop sensors with <= 2 distinct values within a condition\n    \"clip_quantiles\": (0.001, 0.999),\n    # data-quality gate (all fitted on training engines)\n    \"op_margin_frac\": 0.5,        # single-condition subsets: settings beyond the training range by this much = unseen\n    \"sensor_range_margin\": 0.5,   # reading beyond the training min/max by 50% of the range = out of range\n    \"sensor_range_persist\": 2,    # ... for at least this many consecutive cycles\n    \"sensor_stuck_cycles\": 15,    # identical reading this long on a sensor that never flatlines in training = stuck\n    \"roll_window\": 10, \"slope_window\": 20, \"ewm_span\": 10, \"baseline_cycles\": 10,\n    # splits (by engine)\n    \"val_frac\": 0.20, \"new_frac\": 0.30,\n    \"cv_folds\": 5,\n    # 'test-like' evaluation rows: mirror how the official test engines are cut.\n    # Test RUL reaches ~195 in FD002/FD004, so the window covers RUL up to 200.\n    \"eval_min_cycles\": 31, \"eval_max_rul\": 200,\n    # anomaly detector: trained on early-life cycles (no failure labels needed; a real fleet would use the\n    # first cycles after commissioning or maintenance). Starts after the rolling window is full.\n    \"anomaly_train_cycles\": (11, 40), \"anomaly_quantile\": 0.99, \"anomaly_persist\": 3,\n    \"anomaly_healthy_rul\": 125,  # evaluation only: false-alarm rate is measured on RUL > 125 (a proxy)\n    # RUL interval: conformal calibration of the 10% / 90% quantile models\n    \"interval_coverage\": 0.80, \"review_width_quantile\": 0.90,\n    # retraining / promotion\n    \"retrain_min_new_frac\": 0.20, \"psi_alert\": 0.20, \"promote_max_nasa_ratio\": 1.10,\n    \"promote_min_prob\": 0.80,     # how sure the bootstrap must be (80% of resamples of the validation engines)\n    # retraining here is always \"same design, more data\", so v2 is promoted unless it is clearly WORSE\n    # (non-inferiority); \"superiority\" (v2 must be clearly better) is the stricter test for design changes\n    \"promotion_test\": \"non_inferiority\",\n    \"noninferiority_margin\": 0.5,  # cycles of capped RMSE: \"clearly worse\" means worse by more than this (< fold spread)\n    # after promotion, retrain the winning design on every training engine and deploy that\n    \"full_refit\": False,\n}\n\nON_KAGGLE = Path(\"/kaggle/input\").exists()\nWORK = Path(\"/kaggle/working\") if ON_KAGGLE else Path.cwd() / \"outputs\"\nOUT = WORK / \"artifacts\" / f\"{SUBSET}{RUN_TAG}\"\n(OUT / \"plots\").mkdir(parents=True, exist_ok=True)\n(OUT / \"models\").mkdir(parents=True, exist_ok=True)\nnp.random.seed(SEED)\n\n\ndef gpu_available():\n    try:\n        r = subprocess.run([\"nvidia-smi\", \"-L\"], capture_output=True, text=True)\n        return r.returncode == 0 and \"GPU\" in r.stdout\n    except FileNotFoundError:\n        return False\n\n\ndef now_iso():\n    return datetime.now(timezone.utc).isoformat(timespec=\"seconds\")\n\n\ndef json_default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, (Path,)):\n        return str(o)\n    raise TypeError(type(o))\n\n\ndef save_json(obj, path):\n    Path(path).write_text(json.dumps(obj, indent=2, default=json_default))\n\n\nGPU = gpu_available()\nprint(f\"subset={SUBSET}  gpu={GPU}  xgboost={xgb.__version__ if xgb else 'not installed'}  outputs={OUT}\")",
"COLS = [\"unit\", \"cycle\", \"op1\", \"op2\", \"op3\"] + [f\"s{i}\" for i in range(1, 22)]\nOPS, SENSORS = COLS[2:5], COLS[5:]\n\n\ndef find_data_dir(subset):\n    roots = [Path(\"/kaggle/input\"), Path.cwd() / \"data\" / \"csv\", Path.cwd().parent / \"data\" / \"csv\",\n             Path.cwd() / \"DataSet\", Path.cwd().parent / \"DataSet\"]\n    for root in roots:\n        if root.exists():\n            for ext in (\"csv\", \"txt\"):\n                hits = sorted(root.rglob(f\"train_{subset}.{ext}\"))\n                if hits:\n                    return hits[0].parent, ext\n    raise FileNotFoundError(f\"train_{subset}.csv / .txt not found under {[str(r) for r in roots]}\")\n\n\ndef read_table(path, ext):\n    if ext == \"csv\":\n        return pd.read_csv(path)\n    df = pd.read_csv(path, sep=r\"\\s+\", header=None)\n    return df.iloc[:, :len(COLS)].set_axis(COLS, axis=1)\n\n\ndef validate_raw(df, name):\n    missing = [c for c in COLS if c not in df.columns]\n    if missing:\n        raise ValueError(f\"{name}: missing columns {missing}\")\n    num = df[COLS].apply(pd.to_numeric, errors=\"coerce\")\n    report = {\n        \"rows\": len(df),\n        \"engines\": int(num.unit.nunique()),\n        \"missing_cells\": int(df[COLS].isna().sum().sum()),\n        \"non_numeric_cells\": int(num.isna().sum().sum() - df[COLS].isna().sum().sum()),\n        \"duplicate_unit_cycle\": int(num.duplicated(subset=[\"unit\", \"cycle\"]).sum()),\n        \"cycle_gaps\": int(num.sort_values([\"unit\", \"cycle\"]).groupby(\"unit\").cycle.diff().dropna().ne(1).sum()),\n    }\n    if report[\"rows\"] == 0:\n        raise ValueError(f\"{name}: empty file\")\n    return report\n\n\ndef load_subset(subset):\n    d, ext = find_data_dir(subset)\n    train = read_table(d / f\"train_{subset}.{ext}\", ext)\n    test = read_table(d / f\"test_{subset}.{ext}\", ext)\n    if ext == \"csv\":\n        r = pd.read_csv(d / f\"RUL_{subset}.csv\")\n        final = (r.sort_values(\"unit\") if \"unit\" in r else r)[\"rul\"].to_numpy()\n    else:\n        final = pd.read_csv(d / f\"RUL_{subset}.txt\", sep=r\"\\s+\", header=None)[0].to_numpy()\n    reports = {\"train\": validate_raw(train, \"train\"), \"test\": validate_raw(test, \"test\")}\n    test_units = np.sort(pd.to_numeric(test.unit).unique())\n    if len(final) != len(test_units):\n        raise ValueError(f\"RUL file has {len(final)} values but test has {len(test_units)} engines\")\n    rul_final = pd.DataFrame({\"unit\": test_units.astype(int), \"rul_final\": final.astype(int)})\n    return train[COLS], test[COLS], rul_final, reports, d\n\n\nraw_train, raw_test, rul_final, ingest_report, DATA_DIR = load_subset(SUBSET)\nprint(\"data dir:\", DATA_DIR)\ndisplay(pd.DataFrame(ingest_report).T)",
"def basic_clean(df):\n    df = df[COLS].apply(pd.to_numeric, errors=\"coerce\").dropna(subset=[\"unit\", \"cycle\"])\n    df[[\"unit\", \"cycle\"]] = df[[\"unit\", \"cycle\"]].astype(int)\n    df = df.drop_duplicates(subset=[\"unit\", \"cycle\"], keep=\"last\").sort_values([\"unit\", \"cycle\"]).reset_index(drop=True)\n    vals = OPS + SENSORS\n    df[\"imputed\"] = df[vals].isna().any(axis=1)\n    # past-only: forward fill within each engine; leading gaps become the training mean after normalisation\n    df[vals] = df.groupby(\"unit\")[vals].ffill()\n    return df\n\n\ndef add_labels(df, cfg, rul_final=None):\n    df = df.copy()\n    last = df.groupby(\"unit\").cycle.transform(\"max\")\n    end = 0 if rul_final is None else df.unit.map(rul_final.set_index(\"unit\").rul_final)\n    df[\"rul\"] = (end + last - df.cycle).astype(int)\n    df[\"rul_capped\"] = df.rul.clip(upper=cfg[\"rul_cap\"])\n    df[\"fail\"] = (df.rul <= cfg[\"fail_window\"]).astype(int)\n    return df\n\n\ntrain_df = add_labels(basic_clean(raw_train), CFG)\ntest_df = add_labels(basic_clean(raw_test), CFG, rul_final)\n\ncheck = test_df.groupby(\"unit\").rul.min().reset_index().merge(rul_final, on=\"unit\")\nassert (check.rul == check.rul_final).all(), \"test labels do not match the RUL file\"\nprint(f\"train: {train_df.unit.nunique()} engines, {len(train_df):,} rows | test: {test_df.unit.nunique()} engines, {len(test_df):,} rows\")\nprint(f\"imputed rows: train {int(train_df.imputed.sum())}, test {int(test_df.imputed.sum())}\")\nprint(f\"class balance (fail = RUL <= {CFG['fail_window']}): {train_df.fail.mean():.1%} positive\")",
"META = [\"unit\", \"cycle\", \"condition\", \"imputed\", \"unseen_condition\", \"sensor_fault\", \"sensor_fault_detail\",\n        \"rul\", \"rul_capped\", \"fail\"]\n\n\ndef condition_key(df):\n    return (df[OPS].round(0) + 0.0).astype(str).agg(\"/\".join, axis=1)\n\n\ndef assign_condition(df, prep):\n    # operating condition per row; settings never seen in training fall back to the nearest known condition and\n    # are flagged, instead of silently using global statistics\n    if not prep[\"condition_norm\"]:\n        lo = pd.Series({o: prep[\"op_range\"][o][0] for o in OPS})\n        hi = pd.Series({o: prep[\"op_range\"][o][1] for o in OPS})\n        margin = (hi - lo) * prep[\"quality\"][\"op_margin_frac\"] + 0.01 * np.maximum(1.0, np.maximum(lo.abs(), hi.abs()))\n        unseen = ((df[OPS] < lo - margin) | (df[OPS] > hi + margin)).any(axis=1)\n        return pd.Series(\"all\", index=df.index), unseen\n    key = condition_key(df)\n    unseen = ~key.isin(list(prep[\"stats\"]))\n    if unseen.any():\n        names = list(prep[\"centers\"])\n        centers = np.array([prep[\"centers\"][n] for n in names], dtype=float)\n        scale = np.where(centers.std(axis=0) > 0, centers.std(axis=0), 1.0)\n        x = df.loc[unseen, OPS].to_numpy(float)\n        dist = np.nansum(((x[:, None, :] - centers[None]) / scale) ** 2, axis=2)\n        nearest = np.array(names, dtype=object)[dist.argmin(axis=1)]\n        nearest[np.isnan(x).all(axis=1)] = prep[\"default_condition\"]\n        key = key.copy()\n        key.loc[unseen] = nearest\n    return key, unseen\n\n\ndef _flat(frame, units, window):\n    # identical for the whole window: max - min is exactly 0 (a rolling std leaves rounding residue up to ~1e-5)\n    return (_rolling_by_unit(frame, units, window, \"max\") - _rolling_by_unit(frame, units, window, \"min\")) == 0\n\n\ndef _rolling_by_unit(frame, units, window, how):\n    # per-engine trailing window statistic, aligned back to the frame's row order\n    r = frame.groupby(units.to_numpy()).rolling(window, min_periods=window)\n    return getattr(r, how)().reset_index(level=0, drop=True).reindex(frame.index)\n\n\ndef sensor_quality(df, prep):\n    # past-only sensor-health checks: dead (never reported), stuck (flat for K cycles on a sensor that never\n    # flatlines in training) and out of the physically seen range for >= persist cycles\n    q, sensors = prep[\"quality\"], prep[\"sensors\"]\n    raw = df[sensors].astype(float)\n    lo = pd.DataFrame(prep[\"sensor_range\"][\"lo\"]).T.loc[df[\"condition\"], sensors].to_numpy()\n    hi = pd.DataFrame(prep[\"sensor_range\"][\"hi\"]).T.loc[df[\"condition\"], sensors].to_numpy()\n    span, x = hi - lo, raw.to_numpy()\n    oor = (x < lo - q[\"range_margin\"] * span) | (x > hi + q[\"range_margin\"] * span)\n    # an unseen operating condition shifts every reading; that is flagged separately, not blamed on the sensors\n    oor &= ~df[\"unseen_condition\"].to_numpy()[:, None]\n    oor = pd.DataFrame(oor, index=df.index, columns=sensors).astype(float)\n    oor = _rolling_by_unit(oor, df[\"unit\"], q[\"range_persist\"], \"min\").fillna(0).astype(bool)\n    stuck = pd.DataFrame(False, index=df.index, columns=sensors)\n    if prep[\"stuck_sensors\"]:\n        stuck[prep[\"stuck_sensors\"]] = _flat(raw[prep[\"stuck_sensors\"]], df[\"unit\"], q[\"stuck_cycles\"])\n    dead = raw.isna()\n    fault = (oor | stuck | dead).any(axis=1)\n    detail = pd.Series(\"\", index=df.index)\n    for i in np.flatnonzero(fault.to_numpy()):\n        row = df.index[i]\n        parts = [f\"{s} dead\" for s in sensors if dead.at[row, s]] + \\\n                [f\"{s} stuck\" for s in sensors if stuck.at[row, s]] + \\\n                [f\"{s} out of range\" for s in sensors if oor.at[row, s]]\n        detail.at[row] = \"; \".join(parts)\n    return fault, detail\n\n\ndef _normalise(df, prep, sensors):\n    out = pd.DataFrame(index=df.index, columns=sensors, dtype=float)\n    for c, g in df.groupby(\"condition\"):\n        st_ = prep[\"stats\"].get(c, prep[\"global\"])\n        mu = pd.Series(st_[\"mean\"])[sensors]\n        sd = pd.Series(st_[\"std\"])[sensors].replace(0, np.nan)\n        out.loc[g.index] = ((g[sensors] - mu) / sd).to_numpy()\n    return out.fillna(0.0)\n\n\ndef fit_prep(df, cfg):\n    df = df.assign(condition=condition_key(df) if cfg[\"condition_norm\"] else \"all\")\n    stats = {c: {\"mean\": g[SENSORS].mean().to_dict(), \"std\": g[SENSORS].std().to_dict()}\n             for c, g in df.groupby(\"condition\")}\n    glob = {\"mean\": df[SENSORS].mean().to_dict(), \"std\": df[SENSORS].std().to_dict()}\n    nunique = df.groupby(\"condition\")[SENSORS].nunique().median()\n    sensors = [s for s in SENSORS if nunique[s] > cfg[\"min_unique\"]]\n    prep = {\"sensors\": sensors, \"stats\": stats, \"global\": glob, \"condition_norm\": cfg[\"condition_norm\"]}\n    z = _normalise(df, prep, sensors)\n    lo, hi = cfg[\"clip_quantiles\"]\n    prep[\"clip\"] = {s: (float(z[s].quantile(lo)), float(z[s].quantile(hi))) for s in sensors}\n    prep[\"dropped\"] = [s for s in SENSORS if s not in sensors]\n    # what \"normal\" looks like, for the unseen-condition check and the sensor-health gate\n    grouped = df.groupby(\"condition\")\n    prep[\"centers\"] = {c: [float(v) for v in g[OPS].mean()] for c, g in grouped}\n    prep[\"default_condition\"] = str(df[\"condition\"].value_counts().idxmax())\n    prep[\"op_range\"] = {o: [float(df[o].min()), float(df[o].max())] for o in OPS}\n    prep[\"sensor_range\"] = {\"lo\": grouped[sensors].min().T.to_dict(), \"hi\": grouped[sensors].max().T.to_dict()}\n    k = cfg[\"sensor_stuck_cycles\"]\n    flat = _flat(df[sensors].astype(float), df[\"unit\"], k)\n    prep[\"stuck_sensors\"] = [s for s in sensors if flat[s].mean() < 1e-4]  # never flat in training -> checkable\n    prep[\"quality\"] = {\"op_margin_frac\": cfg[\"op_margin_frac\"], \"range_margin\": cfg[\"sensor_range_margin\"],\n                       \"range_persist\": cfg[\"sensor_range_persist\"], \"stuck_cycles\": k}\n    return prep\n\n\ndef apply_prep(df, prep):\n    df = df.sort_values([\"unit\", \"cycle\"]).reset_index(drop=True)\n    condition, unseen = assign_condition(df, prep)\n    df = df.assign(condition=condition, unseen_condition=unseen.to_numpy())\n    fault, detail = sensor_quality(df, prep)\n    df = df.assign(sensor_fault=fault.to_numpy(), sensor_fault_detail=detail.to_numpy())\n    sensors = prep[\"sensors\"]\n    z = _normalise(df, prep, sensors)\n    for s in sensors:\n        z[s] = z[s].clip(*prep[\"clip\"][s])\n    return pd.concat([df[[c for c in META if c in df]], z], axis=1)\n\n\ndef _rolling_sum(a, w):\n    c = np.cumsum(a, axis=0)\n    out = c.copy()\n    out[w:] = c[w:] - c[:-w]\n    return out\n\n\ndef _rolling_slope(y, w):\n    # least-squares slope over the last w cycles (fewer at the start of an engine's life)\n    n_rows = len(y)\n    t = np.arange(n_rows, dtype=float)[:, None]\n    n = np.minimum(np.arange(1, n_rows + 1), w)[:, None].astype(float)\n    st, stt = _rolling_sum(t, w), _rolling_sum(t * t, w)\n    sy, sty = _rolling_sum(y, w), _rolling_sum(t * y, w)\n    den = n * stt - st ** 2\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        slope = (n * sty - st * sy) / den\n    return np.where(den > 1e-9, slope, 0.0)\n\n\ndef feature_cols(prep, cfg):\n    w, sw, span = cfg[\"roll_window\"], cfg[\"slope_window\"], cfg[\"ewm_span\"]\n    sufs = [\"\", f\"_mean{w}\", f\"_std{w}\", f\"_slope{sw}\", f\"_ewm{span}\", \"_drift\"]\n    return [\"cycle\"] + [f\"{s}{suf}\" for suf in sufs for s in prep[\"sensors\"]]\n\n\ndef anomaly_cols(prep, cfg):\n    return [f\"{s}_mean{cfg['roll_window']}\" for s in prep[\"sensors\"]]\n\n\ndef build_features(dz, sensors, cfg):\n    w, sw, span, base_n = cfg[\"roll_window\"], cfg[\"slope_window\"], cfg[\"ewm_span\"], cfg[\"baseline_cycles\"]\n    blocks = []\n    for _, g in dz.groupby(\"unit\", sort=True):\n        y = g[sensors].to_numpy(float)\n        frame = pd.DataFrame(y)\n        roll = frame.rolling(w, min_periods=1)\n        ewm = frame.ewm(span=span, adjust=False).mean().to_numpy()\n        base = np.cumsum(y, axis=0) / np.arange(1, len(y) + 1)[:, None]\n        if len(y) > base_n:\n            base[base_n:] = base[base_n - 1]\n        blocks.append(np.hstack([y, roll.mean().to_numpy(), roll.std().fillna(0.0).to_numpy(),\n                                 _rolling_slope(y, sw), ewm, ewm - base]))\n    names = feature_cols({\"sensors\": sensors}, cfg)[1:]\n    X = pd.DataFrame(np.vstack(blocks), columns=names, index=dz.index)\n    return pd.concat([dz[[c for c in META if c in dz]], X], axis=1)\n\n\ndef featurize(df, prep, cfg):\n    return build_features(apply_prep(df, prep), prep[\"sensors\"], cfg)\n\n\ndef check_causality(df, prep, cfg, n_units=8, seed=SEED):\n    rng = np.random.default_rng(seed)\n    full = featurize(df, prep, cfg).set_index([\"unit\", \"cycle\"])\n    feats = feature_cols(prep, cfg)[1:]\n    for u in rng.choice(df.unit.unique(), n_units, replace=False):\n        g = df[df.unit == u]\n        cut = int(rng.integers(1, len(g) + 1))\n        part = featurize(g.iloc[:cut], prep, cfg).iloc[-1]\n        ref = full.loc[(u, int(part.cycle)), feats].to_numpy(float)\n        assert np.allclose(part[feats].to_numpy(float), ref, atol=1e-8), f\"feature leak at engine {u}\"\n    return True\n\n\ndef check_cleaning_causality(raw, prep, cfg, n_units=8, seed=SEED):\n    # inject gaps (dense at the start of the series), then check cleaning + features stay past-only\n    rng = np.random.default_rng(seed)\n    feats = feature_cols(prep, cfg)[1:]\n    for u in rng.choice(raw.unit.unique(), n_units, replace=False):\n        g = raw[raw.unit == u].copy()\n        gaps = rng.random((len(g), len(SENSORS))) < 0.05\n        gaps[:3] |= rng.random((3, len(SENSORS))) < 0.5\n        g[SENSORS] = g[SENSORS].mask(gaps)\n        full = featurize(basic_clean(g), prep, cfg).set_index(\"cycle\")\n        cut = int(rng.integers(1, len(g) + 1))\n        part = featurize(basic_clean(g.iloc[:cut]), prep, cfg).iloc[-1]\n        ref = full.loc[int(part.cycle), feats].to_numpy(float)\n        assert np.allclose(part[feats].to_numpy(float), ref, atol=1e-8), f\"cleaning leak at engine {u}\"\n    return True\n\n\n_prep_demo = fit_prep(train_df, CFG)\n_feat_demo = featurize(train_df, _prep_demo, CFG)\nprint(\"sensors kept   :\", _prep_demo[\"sensors\"])\nprint(\"sensors dropped:\", _prep_demo[\"dropped\"])\nprint(\"conditions     :\", len(_prep_demo[\"stats\"]))\nprint(f\"features       : {len(feature_cols(_prep_demo, CFG))} per row, matrix {_feat_demo.shape}\")\nassert not _feat_demo[feature_cols(_prep_demo, CFG)].isna().any().any()\nprint(\"causality check:\", check_causality(train_df, _prep_demo, CFG))\nprint(\"cleaning causality check (injected gaps):\", check_cleaning_causality(raw_train, _prep_demo, CFG))",
"def split_units(df, cfg, seed=SEED):\n    units = np.array(sorted(df.unit.unique()))\n    np.random.default_rng(seed).shuffle(units)\n    n_val, n_new = round(len(units) * cfg[\"val_frac\"]), round(len(units) * cfg[\"new_frac\"])\n    return {\"val\": sorted(units[:n_val].tolist()),\n            \"new\": sorted(units[n_val:n_val + n_new].tolist()),\n            \"initial\": sorted(units[n_val + n_new:].tolist())}\n\n\nSPLITS = split_units(train_df, CFG)\nassert not (set(SPLITS[\"val\"]) & set(SPLITS[\"new\"]) | set(SPLITS[\"val\"]) & set(SPLITS[\"initial\"])\n            | set(SPLITS[\"new\"]) & set(SPLITS[\"initial\"])), \"engine leak between splits\"\nval_df = train_df[train_df.unit.isin(SPLITS[\"val\"])].reset_index(drop=True)\nprint({k: len(v) for k, v in SPLITS.items()}, \"engines\")",
"class Tracker:\n    def __init__(self, root, experiment):\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.run_id = None\n        try:\n            import mlflow\n            from mlflow.tracking import MlflowClient\n            mlflow.set_tracking_uri(f\"sqlite:///{(self.root / 'mlflow.db').as_posix()}\")\n            if mlflow.get_experiment_by_name(experiment) is None:\n                mlflow.create_experiment(experiment, artifact_location=(self.root / \"mlruns\").as_uri())\n            mlflow.set_experiment(experiment)\n            self.mlflow, self.client = mlflow, MlflowClient()\n            print(\"tracking: MLflow\", mlflow.__version__, \"->\", self.root)\n        except Exception as e:\n            self.mlflow = None\n            print(\"tracking: JSON fallback ->\", self.root, f\"({type(e).__name__}: {e})\")\n\n    def _load(self, name, default):\n        p = self.root / name\n        return json.loads(p.read_text()) if p.exists() else default\n\n    def _save(self, name, obj):\n        save_json(obj, self.root / name)\n\n    @contextmanager\n    def run(self, name, **tags):\n        tags = {k: str(v) for k, v in tags.items()}\n        if self.mlflow:\n            with self.mlflow.start_run(run_name=name, tags=tags) as r:\n                self.run_id = r.info.run_id\n                yield self\n        else:\n            self.run_id = f\"{name}-{int(time.time() * 1000)}\"\n            self._rec = {\"run_id\": self.run_id, \"name\": name, \"tags\": tags, \"params\": {}, \"metrics\": {},\n                         \"artifacts\": [], \"start\": now_iso()}\n            try:\n                yield self\n            finally:\n                runs = self._load(\"runs.json\", [])\n                runs.append(self._rec)\n                self._save(\"runs.json\", runs)\n        self.run_id = None\n\n    def log_params(self, params):\n        params = {k: str(v)[:500] for k, v in params.items()}\n        if self.mlflow:\n            self.mlflow.log_params(params)\n        else:\n            self._rec[\"params\"].update(params)\n\n    def log_metrics(self, metrics):\n        metrics = {k: float(v) for k, v in metrics.items()\n                   if isinstance(v, (int, float, np.number)) and np.isfinite(v)}\n        if self.mlflow:\n            self.mlflow.log_metrics(metrics)\n        else:\n            self._rec[\"metrics\"].update(metrics)\n\n    def log_artifact(self, path):\n        if self.mlflow:\n            self.mlflow.log_artifact(str(path))\n        else:\n            self._rec[\"artifacts\"].append(str(path))\n\n    def log_model(self, model, name):\n        if self.mlflow:\n            # XGBoost gets its native flavor; recent MLflow saves sklearn models with skops,\n            # which rejects non-sklearn types, so sklearn models are pinned to cloudpickle\n            if xgb is not None and isinstance(model, xgb.XGBModel):\n                import mlflow.xgboost as flavor\n                kwargs = {}\n            else:\n                import mlflow.sklearn as flavor\n                kwargs = {\"serialization_format\": \"cloudpickle\"}\n            try:\n                info = flavor.log_model(model, name=name, **kwargs)\n            except TypeError:\n                info = flavor.log_model(model, artifact_path=name, **kwargs)\n            return info.model_uri\n        p = self.root / \"models\" / f\"{self.run_id}_{name}.joblib\"\n        p.parent.mkdir(parents=True, exist_ok=True)\n        joblib.dump(model, p)\n        self._rec[\"artifacts\"].append(str(p))\n        return str(p)\n\n    def register(self, model_uri, name, **tags):\n        if self.mlflow:\n            mv = self.mlflow.register_model(model_uri, name)\n            for k, v in tags.items():\n                self.client.set_model_version_tag(name, mv.version, k, str(v))\n            return int(mv.version)\n        reg = self._load(\"registry.json\", {})\n        entry = reg.setdefault(name, {\"versions\": [], \"aliases\": {}})\n        version = len(entry[\"versions\"]) + 1\n        entry[\"versions\"].append({\"version\": version, \"uri\": model_uri, \"created_at\": now_iso(), \"tags\": tags})\n        self._save(\"registry.json\", reg)\n        return version\n\n    def set_alias(self, name, alias, version):\n        if self.mlflow:\n            self.client.set_registered_model_alias(name, alias, str(version))\n        else:\n            reg = self._load(\"registry.json\", {})\n            reg[name][\"aliases\"][alias] = version\n            self._save(\"registry.json\", reg)\n\n    def get_alias(self, name, alias):\n        try:\n            if self.mlflow:\n                return int(self.client.get_model_version_by_alias(name, alias).version)\n            return self._load(\"registry.json\", {}).get(name, {}).get(\"aliases\", {}).get(alias)\n        except Exception:\n            return None\n\n\ntracker = Tracker(WORK / f\"tracking{RUN_TAG}\", experiment=f\"cmapss_{SUBSET}{RUN_TAG}\")\nREGISTRY_NAME = f\"rul_{SUBSET.lower()}{RUN_TAG}\"",
"def make_rul_model(kind):\n    if kind == \"ridge\":\n        return make_pipeline(StandardScaler(), Ridge(alpha=1.0))\n    if kind == \"hgb\":\n        return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,\n                                             l2_regularization=1.0, early_stopping=False, random_state=SEED)\n    if kind == \"xgb\":\n        return xgb.XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=5, subsample=0.8,\n                                colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, tree_method=\"hist\",\n                                device=\"cuda\" if GPU else \"cpu\", random_state=SEED, n_jobs=-1)\n    raise ValueError(kind)\n\n\ndef make_quantile(q):\n    return HistGradientBoostingRegressor(loss=\"quantile\", quantile=q, max_iter=300, learning_rate=0.05,\n                                         max_leaf_nodes=31, min_samples_leaf=20, early_stopping=False,\n                                         random_state=SEED)\n\n\ndef make_classifier():\n    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,\n                                          class_weight=\"balanced\", early_stopping=False, random_state=SEED)\n\n\nCANDIDATES = [\"ridge\", \"hgb\"] + ([\"xgb\"] if xgb is not None else [])\n\n\ndef nasa_score(y_true, y_pred):\n    d = np.asarray(y_pred, float) - np.asarray(y_true, float)\n    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)\n\n\ndef rul_metrics(y_true, y_pred, cap):\n    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)\n    d = y_pred - y_true\n    s = nasa_score(y_true, y_pred)\n    s_cap = nasa_score(np.minimum(y_true, cap), y_pred)\n    return {\"rmse\": float(np.sqrt(np.mean(d ** 2))),\n            \"rmse_capped\": float(np.sqrt(np.mean((y_pred - np.minimum(y_true, cap)) ** 2))),\n            \"mae\": float(np.mean(np.abs(d))), \"bias\": float(np.mean(d)),\n            \"late_pct\": float(np.mean(d > 0) * 100),\n            \"nasa_score\": float(s.sum()), \"nasa_mean\": float(s.mean()),\n            \"nasa_capped_mean\": float(s_cap.mean()), \"n\": int(len(d))}\n\n\ndef clf_metrics(y, p, threshold):\n    y, p = np.asarray(y), np.asarray(p)\n    pred = p >= threshold\n    tp, fp, fn = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum()), int((~pred & (y == 1)).sum())\n    prec = tp / (tp + fp) if tp + fp else 0.0\n    rec = tp / (tp + fn) if tp + fn else 0.0\n    two_classes = len(np.unique(y)) == 2\n    return {\"pr_auc\": float(average_precision_score(y, p)) if two_classes else float(\"nan\"),\n            \"roc_auc\": float(roc_auc_score(y, p)) if two_classes else float(\"nan\"),\n            \"precision\": prec, \"recall\": rec,\n            \"f2\": 5 * prec * rec / (4 * prec + rec) if prec + rec else 0.0,\n            \"positive_rate\": float(y.mean())}\n\n\ndef anomaly_metrics(P, cfg):\n    healthy = P.rul > cfg[\"anomaly_healthy_rul\"]\n    near = P.rul <= cfg[\"fail_window\"]\n    alarms = P[P.anomaly_persistent & ~healthy]\n    first = alarms.groupby(\"unit\").rul.max()\n    return {\"false_alarm_rate_healthy\": float(P.loc[healthy, \"anomaly_persistent\"].mean()) if healthy.any() else float(\"nan\"),\n            \"detection_rate_last30\": float(P.loc[near, \"anomaly_persistent\"].mean()) if near.any() else float(\"nan\"),\n            \"engines_detected_pct\": float(len(first) / P.unit.nunique() * 100),\n            \"median_rul_at_first_alarm\": float(first.median()) if len(first) else float(\"nan\")}\n\n\ndef interval_coverage(P):\n    # quantile models are trained on the capped target, so coverage is measured against it\n    return float(((P.rul_capped >= P.rul_lo) & (P.rul_capped <= P.rul_hi)).mean())\n\n\ndef coverage_by_band(P, name, bands=((0, 30), (30, 60), (60, 125))):\n    # coverage where decisions are made; CRITICAL relies on the lower bound when RUL <= 30\n    out = {}\n    for lo, hi in bands:\n        b = P[(P.rul > lo) & (P.rul <= hi)]\n        if len(b):\n            out[f\"{name}_coverage_rul_{lo}_{hi}\"] = interval_coverage(b)\n            out[f\"{name}_below_lower_bound_rul_{lo}_{hi}\"] = float((b.rul_capped < b.rul_lo).mean())\n    return out\n\n\ndef bootstrap_gain(P_old, P_new, cfg, n_boot=2000, seed=SEED):\n    # resample validation engines: how often is the new model better on capped RMSE, and how often is its\n    # capped NASA score more than the allowed ratio worse?\n    cap = cfg[\"rul_cap\"]\n    def per_engine(P):\n        P = test_like(P, cfg)\n        y = np.minimum(P.rul, cap)\n        return P.assign(se=(P.rul_pred - y) ** 2, nasa=nasa_score(y, P.rul_pred)).groupby(\"unit\")[[\"se\", \"nasa\"]] \\\n                .agg([\"sum\", \"count\"])\n    a = per_engine(P_old)\n    b = per_engine(P_new).reindex(a.index)\n    pick = np.random.default_rng(seed).integers(0, len(a), size=(n_boot, len(a)))\n    mean = lambda s, col: s[(col, \"sum\")].to_numpy()[pick].sum(axis=1) / s[(col, \"count\")].to_numpy()[pick].sum(axis=1)\n    diff = np.sqrt(mean(b, \"se\")) - np.sqrt(mean(a, \"se\"))\n    nasa_ratio = mean(b, \"nasa\") / np.maximum(mean(a, \"nasa\"), 1e-12)\n    return {\"prob_new_better\": float((diff < 0).mean()), \"prob_new_worse\": float((diff > 0).mean()),\n            \"prob_new_worse_by_margin\": float((diff > cfg[\"noninferiority_margin\"]).mean()),\n            \"rmse_diff_ci90\": [float(np.quantile(diff, 0.05)), float(np.quantile(diff, 0.95))],\n            \"prob_nasa_clearly_worse\": float((nasa_ratio > cfg[\"promote_max_nasa_ratio\"]).mean()),\n            \"nasa_ratio_ci90\": [float(np.quantile(nasa_ratio, 0.05)), float(np.quantile(nasa_ratio, 0.95))]}\n\n\ndef conformal_correction(y, lo, hi, coverage):\n    # conformalized quantile regression: how far outside [lo, hi] the truth falls, at the target coverage\n    scores = np.maximum(np.asarray(lo) - np.asarray(y), np.asarray(y) - np.asarray(hi))\n    level = min(1.0, np.ceil((len(scores) + 1) * coverage) / len(scores))\n    return float(np.quantile(scores, level))\n\n\ndef prefix(p, d):\n    return {f\"{p}_{k}\": v for k, v in d.items()}\n\n\ndef latest(P):\n    return P.sort_values([\"unit\", \"cycle\"]).groupby(\"unit\").tail(1).reset_index(drop=True)\n\n\ndef test_like(P, cfg):\n    return P[(P.cycle >= cfg[\"eval_min_cycles\"]) & (P.rul <= cfg[\"eval_max_rul\"])]\n\n\ndef train_bundle(df, kind, cfg, clf_threshold, interval_correction=0.0):\n    t0 = time.time()\n    prep = fit_prep(df, cfg)\n    F = featurize(df, prep, cfg)\n    feats, acols = feature_cols(prep, cfg), anomaly_cols(prep, cfg)\n    X, y = F[feats], F.rul_capped\n    first, last = cfg[\"anomaly_train_cycles\"]\n    healthy = F.cycle.between(first, last)  # early life: no failure labels used\n    iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(F.loc[healthy, acols])\n    return {\n        \"kind\": kind, \"subset\": cfg[\"subset\"], \"cfg\": dict(cfg), \"prep\": prep,\n        \"features\": feats, \"anomaly_features\": acols,\n        \"rul_model\": make_rul_model(kind).fit(X, y),\n        \"q_lo\": make_quantile(0.1).fit(X, y), \"q_hi\": make_quantile(0.9).fit(X, y),\n        \"interval_correction\": float(interval_correction),\n        \"classifier\": make_classifier().fit(X, F.fail), \"clf_threshold\": float(clf_threshold),\n        \"iso\": iso,\n        \"anomaly_threshold\": float(np.quantile(-iso.score_samples(F.loc[healthy, acols]), cfg[\"anomaly_quantile\"])),\n        \"units\": sorted(int(u) for u in df.unit.unique()), \"n_rows\": int(len(df)),\n        \"train_seconds\": round(time.time() - t0, 1), \"created_at\": now_iso(),\n    }\n\n\ndef predict_bundle(b, df):\n    cfg, cap = b[\"cfg\"], b[\"cfg\"][\"rul_cap\"]\n    F = featurize(df, b[\"prep\"], cfg)\n    X = F[b[\"features\"]]\n    out = F[[c for c in META if c in F]].copy()\n    pred = np.clip(b[\"rul_model\"].predict(X), 0, cap)\n    q = b.get(\"interval_correction\", 0.0)\n    out[\"rul_pred\"] = pred\n    out[\"rul_lo\"] = np.minimum(np.clip(b[\"q_lo\"].predict(X) - q, 0, cap), pred)\n    out[\"rul_hi\"] = np.maximum(np.clip(b[\"q_hi\"].predict(X) + q, 0, cap), pred)\n    out[\"p_fail\"] = b[\"classifier\"].predict_proba(X)[:, 1]\n    out[\"fail_alert\"] = out.p_fail >= b[\"clf_threshold\"]\n    out[\"anomaly_score\"] = -b[\"iso\"].score_samples(F[b[\"anomaly_features\"]])\n    out[\"anomaly\"] = out.anomaly_score > b[\"anomaly_threshold\"]\n    k, w = cfg[\"anomaly_persist\"], cfg[\"roll_window\"]\n    g = out.groupby(\"unit\")\n    out[\"anomaly_persistent\"] = g.anomaly.transform(lambda a: a.astype(int).rolling(k, min_periods=k).min().fillna(0)).astype(bool)\n    out[\"recent_imputed\"] = g.imputed.transform(lambda a: a.astype(int).rolling(w, min_periods=1).max()).astype(bool)\n    out[\"recent_unseen\"] = g.unseen_condition.transform(lambda a: a.astype(int).rolling(w, min_periods=1).max()).astype(bool)\n    out[\"cycles_observed\"] = g.cumcount() + 1\n    return out\n\n\ndef evaluate_bundle(b, val_df, test_df):\n    # val_df=None skips validation metrics, for a model that has trained on the validation engines\n    cfg, cap = b[\"cfg\"], b[\"cfg\"][\"rul_cap\"]\n    Pt = predict_bundle(b, test_df)\n    last = latest(Pt)\n    m = prefix(\"test\", rul_metrics(last.rul, last.rul_pred, cap))\n    m[\"test_interval_coverage\"] = interval_coverage(last)\n    m.update(coverage_by_band(last, \"test\"))\n    Pv = None\n    if val_df is not None:\n        Pv = predict_bundle(b, val_df)\n        tl = test_like(Pv, cfg)\n        m.update(prefix(\"val\", rul_metrics(tl.rul, tl.rul_pred, cap)))\n        m.update(prefix(\"val_all\", rul_metrics(Pv.rul, Pv.rul_pred, cap)))\n        m[\"val_interval_coverage\"] = interval_coverage(tl)\n        m.update(coverage_by_band(tl, \"val\"))\n        m.update(prefix(\"val_clf\", clf_metrics(Pv.fail, Pv.p_fail, b[\"clf_threshold\"])))\n        m.update(prefix(\"val_anom\", anomaly_metrics(Pv, cfg)))\n    return m, Pv, Pt\n\n\ndef show_metrics(m, keys=(\"val_rmse_capped\", \"val_rmse\", \"val_nasa_capped_mean\", \"test_rmse_capped\", \"test_rmse\",\n                          \"test_nasa_score\", \"test_mae\", \"test_bias\",\n                          \"test_late_pct\", \"val_interval_coverage\", \"test_interval_coverage\", \"val_clf_pr_auc\", \"val_clf_recall\",\n                          \"val_clf_precision\", \"val_anom_false_alarm_rate_healthy\", \"val_anom_median_rul_at_first_alarm\")):\n    return pd.Series({k: round(m[k], 3) for k in keys if k in m})",
"def select_and_calibrate(units, label):\n    # CV over the given engines only: picks the RUL model and sets the classifier, interval and review values\n    dev = train_df[train_df.unit.isin(units)].reset_index(drop=True)\n    cap = CFG[\"rul_cap\"]\n    fold_frames = []\n    t0 = time.time()\n    for fold, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=CFG[\"cv_folds\"]).split(dev, groups=dev.unit)):\n        tr, va = dev.iloc[tr_idx], dev.iloc[va_idx]\n        prep = fit_prep(tr, CFG)\n        Ftr, Fva = featurize(tr, prep, CFG), featurize(va, prep, CFG)\n        feats = feature_cols(prep, CFG)\n        o = Fva[[\"unit\", \"cycle\", \"rul\", \"rul_capped\", \"fail\"]].copy()\n        o[\"fold\"] = fold\n        for kind in CANDIDATES:\n            o[kind] = np.clip(make_rul_model(kind).fit(Ftr[feats], Ftr.rul_capped).predict(Fva[feats]), 0, cap)\n        o[\"p_fail\"] = make_classifier().fit(Ftr[feats], Ftr.fail).predict_proba(Fva[feats])[:, 1]\n        o[\"q_lo\"] = make_quantile(0.1).fit(Ftr[feats], Ftr.rul_capped).predict(Fva[feats])\n        o[\"q_hi\"] = make_quantile(0.9).fit(Ftr[feats], Ftr.rul_capped).predict(Fva[feats])\n        fold_frames.append(o)\n        print(f\"[{label}] fold {fold}: {va.unit.nunique()} engines held out  ({time.time() - t0:.0f}s)\")\n    oof = pd.concat(fold_frames, ignore_index=True)\n\n    oof_tl = test_like(oof, CFG)\n    cv_results = pd.DataFrame({k: rul_metrics(oof_tl.rul, oof_tl[k], cap) for k in CANDIDATES}).T\n    kind = cv_results.rmse_capped.idxmin()\n\n    prec, rec, thr = precision_recall_curve(oof.fail, oof.p_fail)\n    f2 = 5 * prec * rec / np.maximum(4 * prec + rec, 1e-9)\n    clf_threshold = float(thr[np.argmax(f2[:-1])])\n    cv_clf = clf_metrics(oof.fail, oof.p_fail, clf_threshold)\n\n    # Calibrate only on rows with true RUL <= cap: above it the capped truth (125) sits at the clipped upper\n    # bound and is covered almost automatically, which would shrink the margin where it matters.\n    oof_cal = oof_tl[oof_tl.rul <= cap]\n    raw_cover = float(((oof_cal.rul_capped >= oof_cal.q_lo) & (oof_cal.rul_capped <= oof_cal.q_hi)).mean())\n    interval_q = conformal_correction(oof_cal.rul_capped, oof_cal.q_lo, oof_cal.q_hi, CFG[\"interval_coverage\"])\n    lo_cal = np.minimum(np.clip(oof.q_lo - interval_q, 0, cap), oof[kind])\n    hi_cal = np.maximum(np.clip(oof.q_hi + interval_q, 0, cap), oof[kind])\n    cal_cover = float(((oof.rul_capped >= lo_cal) & (oof.rul_capped <= hi_cal)).loc[oof_cal.index].mean())\n    likely_orders = oof[kind] <= 60\n    review_width = float(np.quantile((hi_cal - lo_cal)[likely_orders], CFG[\"review_width_quantile\"]))\n    cv_interval = {\"raw_coverage\": raw_cover, \"calibrated_coverage\": cal_cover,\n                   \"correction_cycles\": interval_q, \"review_width\": review_width}\n\n    for k in CANDIDATES:\n        with tracker.run(f\"{SUBSET}_{label}_cv_{k}\", subset=SUBSET, stage=f\"{label}_model_selection\", model=k):\n            tracker.log_params({\"model\": k, \"folds\": CFG[\"cv_folds\"], \"engines\": len(units), **CFG})\n            tracker.log_metrics(prefix(\"cv\", cv_results.loc[k].to_dict()))\n    with tracker.run(f\"{SUBSET}_{label}_cv_calibration\", subset=SUBSET, stage=f\"{label}_model_selection\"):\n        tracker.log_params({\"clf_threshold\": clf_threshold, \"class_weight\": \"balanced\", \"fail_window\": CFG[\"fail_window\"],\n                            \"target_coverage\": CFG[\"interval_coverage\"], \"review_width_quantile\": CFG[\"review_width_quantile\"]})\n        tracker.log_metrics({**prefix(\"cv_clf\", cv_clf), **prefix(\"cv_interval\", cv_interval)})\n\n    display(cv_results.round(3))\n    print(f\"[{label}] {len(units)} engines | best RUL model: {kind} | classifier threshold (max F2): {clf_threshold:.3f} | \"\n          f\"CV PR-AUC {cv_clf['pr_auc']:.3f}, recall {cv_clf['recall']:.2f}, precision {cv_clf['precision']:.2f}\")\n    print(f\"[{label}] interval: raw coverage {raw_cover:.2f} -> widened by {interval_q:.1f} cycles each side -> \"\n          f\"{cal_cover:.2f} (target {CFG['interval_coverage']:.2f}) | review if width > {review_width:.1f} cycles\")\n    return {\"label\": label, \"engines\": len(units), \"kind\": kind, \"clf_threshold\": clf_threshold,\n            \"interval_q\": interval_q, \"review_width\": review_width,\n            \"cv_results\": cv_results, \"cv_clf\": cv_clf, \"cv_interval\": cv_interval}\n\n\nSEL_V1 = select_and_calibrate(SPLITS[\"initial\"], \"v1\")",
"history = []\n\n\ndef train_and_log(units, label, sel, validate=True, **tags):\n    # sel: the selection/calibration result this version is built with (section 7)\n    data = train_df[train_df.unit.isin(units)].reset_index(drop=True)\n    b = train_bundle(data, sel[\"kind\"], CFG, sel[\"clf_threshold\"], sel[\"interval_q\"])\n    b.update({\"label\": label, \"selection\": sel, \"review_width\": sel[\"review_width\"]})\n    m, Pv, Pt = evaluate_bundle(b, val_df if validate else None, test_df)\n    path = OUT / \"models\" / f\"bundle_{label}.joblib\"\n    with tracker.run(f\"{SUBSET}_{label}\", subset=SUBSET, stage=label, **tags):\n        tracker.log_params({\"model\": sel[\"kind\"], \"train_engines\": len(units), \"train_rows\": len(data),\n                            \"selected_on\": f\"{sel['label']} ({sel['engines']} engines)\",\n                            \"clf_threshold\": sel[\"clf_threshold\"], \"interval_correction\": sel[\"interval_q\"],\n                            \"review_width\": sel[\"review_width\"], **CFG})\n        tracker.log_metrics(m)\n        uri = tracker.log_model(b[\"rul_model\"], \"rul_model\")\n        version = tracker.register(uri, REGISTRY_NAME, bundle=path.name, label=label)\n        b[\"registry_version\"] = version\n        joblib.dump(b, path)\n        tracker.log_artifact(path)\n    history.append({\"version\": version, \"label\": label, \"model\": sel[\"kind\"], \"trained_at\": b[\"created_at\"],\n                    \"train_engines\": len(units), \"train_seconds\": b[\"train_seconds\"], **m})\n    print(f\"{label}: registered {REGISTRY_NAME} v{version}  ({b['train_seconds']}s)\")\n    return b, m, Pv, Pt\n\n\nb1, m1, Pv1, Pt1 = train_and_log(SPLITS[\"initial\"], \"v1_initial\", SEL_V1, trigger=\"initial training\")\ntracker.set_alias(REGISTRY_NAME, \"champion\", b1[\"registry_version\"])\nchampion = {\"bundle\": b1, \"metrics\": m1, \"Pv\": Pv1, \"Pt\": Pt1}\ndisplay(show_metrics(m1).to_frame(\"v1\"))",
"def psi(ref, new, bins=10):\n    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))\n    if len(edges) < 3:\n        return 0.0\n    r = np.histogram(np.clip(ref, edges[0], edges[-1]), edges)[0] / len(ref)\n    n = np.histogram(np.clip(new, edges[0], edges[-1]), edges)[0] / len(new)\n    r, n = np.clip(r, 1e-4, None), np.clip(n, 1e-4, None)\n    return float(np.sum((n - r) * np.log(n / r)))\n\n\n# new data arrives: validate it, then compare it with the data v1 was trained on\nraw_new = raw_train[raw_train.unit.isin(SPLITS[\"new\"])]\nnew_report = validate_raw(raw_new, \"new batch\")\nref_z = apply_prep(train_df[train_df.unit.isin(SPLITS[\"initial\"])], b1[\"prep\"])\nnew_z = apply_prep(train_df[train_df.unit.isin(SPLITS[\"new\"])], b1[\"prep\"])\ndrift = pd.Series({s: psi(ref_z[s], new_z[s]) for s in b1[\"prep\"][\"sensors\"]}).sort_values(ascending=False)\n\nnew_frac = len(SPLITS[\"new\"]) / len(SPLITS[\"initial\"])\ntrigger_reasons = []\nif new_frac >= CFG[\"retrain_min_new_frac\"]:\n    trigger_reasons.append(f\"{len(SPLITS['new'])} new engines = {new_frac:.0%} of training set (>= {CFG['retrain_min_new_frac']:.0%})\")\nif drift.max() > CFG[\"psi_alert\"]:\n    trigger_reasons.append(f\"drift: PSI {drift.max():.2f} on {drift.idxmax()} (> {CFG['psi_alert']})\")\nprint(\"new batch:\", new_report)\nprint(f\"max PSI {drift.max():.3f} ({drift.idxmax()}) | retrain triggered: {bool(trigger_reasons)}\")\nfor r in trigger_reasons:\n    print(\"  -\", r)\n\npromotion = {\"triggered\": bool(trigger_reasons), \"trigger_reasons\": trigger_reasons, \"promoted\": False}\nif trigger_reasons:\n    SEL_V2 = select_and_calibrate(SPLITS[\"initial\"] + SPLITS[\"new\"], \"v2\")\n    b2, m2, Pv2, Pt2 = train_and_log(SPLITS[\"initial\"] + SPLITS[\"new\"], \"v2_retrained\", SEL_V2,\n                                     trigger=\"; \".join(trigger_reasons))\n    better_rmse = m2[\"val_rmse_capped\"] < m1[\"val_rmse_capped\"]\n    boot = bootstrap_gain(Pv1, Pv2, CFG)\n    sure = CFG[\"promote_min_prob\"]\n    if CFG[\"promotion_test\"] == \"non_inferiority\":\n        # same design, more data: promote unless v2 is clearly worse \u2014 RMSE worse by more than the margin, or NASA\n        # more than the allowed ratio worse \u2014 in at least `sure` of the resamples (ties and tiny differences = not worse)\n        robust = boot[\"prob_new_worse_by_margin\"] < sure\n        nasa_ok = boot[\"prob_nasa_clearly_worse\"] < sure\n        promote = robust and nasa_ok\n    else:\n        # superiority (design changes): v2 must be clearly better, and its NASA score within the limit\n        robust = boot[\"prob_new_better\"] >= sure\n        nasa_ok = m2[\"val_nasa_capped_mean\"] <= m1[\"val_nasa_capped_mean\"] * CFG[\"promote_max_nasa_ratio\"]\n        promote = better_rmse and robust and nasa_ok\n    promotion.update({\"challenger_version\": b2[\"registry_version\"], \"champion_version\": b1[\"registry_version\"],\n                      \"val_rmse_capped\": [m1[\"val_rmse_capped\"], m2[\"val_rmse_capped\"]],\n                      \"val_rmse\": [m1[\"val_rmse\"], m2[\"val_rmse\"]],\n                      \"val_nasa_capped_mean\": [m1[\"val_nasa_capped_mean\"], m2[\"val_nasa_capped_mean\"]],\n                      \"val_nasa_mean\": [m1[\"val_nasa_mean\"], m2[\"val_nasa_mean\"]],\n                      \"better_rmse\": better_rmse, \"nasa_within_limit\": nasa_ok,\n                      \"promotion_test\": CFG[\"promotion_test\"],\n                      \"bootstrap_prob_new_better\": boot[\"prob_new_better\"], \"bootstrap_prob_new_worse\": boot[\"prob_new_worse\"],\n                      \"bootstrap_prob_new_worse_by_margin\": boot[\"prob_new_worse_by_margin\"],\n                      \"bootstrap_rmse_diff_ci90\": boot[\"rmse_diff_ci90\"], \"robust_gain\": bool(robust),\n                      \"bootstrap_prob_nasa_clearly_worse\": boot[\"prob_nasa_clearly_worse\"],\n                      \"bootstrap_nasa_ratio_ci90\": boot[\"nasa_ratio_ci90\"],\n                      \"promoted\": bool(promote), \"decided_at\": now_iso()})\n    print(f\"[{CFG['promotion_test']}] v2 better in {boot['prob_new_better']:.0%} of bootstrap resamples \"\n          f\"(RMSE change, 90% CI {boot['rmse_diff_ci90'][0]:+.2f} to {boot['rmse_diff_ci90'][1]:+.2f} cycles); \"\n          f\"NASA more than {CFG['promote_max_nasa_ratio']:.2f}x worse in {boot['prob_nasa_clearly_worse']:.0%}\")\n    if promotion[\"promoted\"]:\n        tracker.set_alias(REGISTRY_NAME, \"champion\", b2[\"registry_version\"])\n        champion = {\"bundle\": b2, \"metrics\": m2, \"Pv\": Pv2, \"Pt\": Pt2}\n    display(pd.DataFrame({\"v1\": show_metrics(m1), \"v2\": show_metrics(m2)}))\n\nwith tracker.run(f\"{SUBSET}_promotion\", subset=SUBSET, stage=\"promotion\"):\n    tracker.log_params({k: v for k, v in promotion.items() if not isinstance(v, list)})\n    tracker.log_metrics(prefix(\"psi\", drift.to_dict()))\n\nCHAMPION_VERSION = champion[\"bundle\"][\"registry_version\"]\nprint(f\"promoted: {promotion['promoted']} -> champion = v{CHAMPION_VERSION} \"\n      f\"(registry alias says v{tracker.get_alias(REGISTRY_NAME, 'champion')})\")",
"if CFG[\"full_refit\"]:\n    all_units = sorted(int(u) for u in train_df.unit.unique())\n    b3, m3, _, Pt3 = train_and_log(all_units, \"v3_full_refit\", champion[\"bundle\"][\"selection\"], validate=False,\n                                   trigger=f\"refit of champion v{CHAMPION_VERSION} design on all {len(all_units)} engines\")\n    production = {\"bundle\": b3, \"metrics\": m3, \"Pt\": Pt3}\nelse:\n    production = {k: champion[k] for k in (\"bundle\", \"metrics\", \"Pt\")}\nPRODUCTION_VERSION = production[\"bundle\"][\"registry_version\"]\ntracker.set_alias(REGISTRY_NAME, \"production\", PRODUCTION_VERSION)\n\ncompare_keys = [\"test_rmse_capped\", \"test_rmse\", \"test_mae\", \"test_bias\", \"test_late_pct\", \"test_nasa_score\", \"test_interval_coverage\"]\nrefit_comparison = pd.DataFrame({\n    f\"champion v{CHAMPION_VERSION} ({len(champion['bundle']['units'])} engines)\": show_metrics(champion[\"metrics\"], compare_keys),\n    f\"production v{PRODUCTION_VERSION} ({len(production['bundle']['units'])} engines)\": show_metrics(production[\"metrics\"], compare_keys),\n})\ndisplay(refit_comparison)\nprint(f\"aliases: champion = v{tracker.get_alias(REGISTRY_NAME, 'champion')}, \"\n      f\"production = v{tracker.get_alias(REGISTRY_NAME, 'production')}\")",
"b_c, m_c, Pt_c = production[\"bundle\"], production[\"metrics\"], production[\"Pt\"]\nPv_c, m_v = champion[\"Pv\"], champion[\"metrics\"]\nfleet = latest(Pt_c)\ndisplay(pd.DataFrame({f\"production v{PRODUCTION_VERSION} (test)\": show_metrics(m_c),\n                      f\"champion v{CHAMPION_VERSION} (validation)\": show_metrics(m_v)}).reindex(show_metrics(m_v).index))\n\n\ndef error_by_band(P, cap, edges=(0, 30, 60, 100, 125, 150, 200)):\n    # where the test error comes from; bands above the cap carry the unavoidable, early-side error\n    d = P.assign(err=P.rul_pred - P.rul, band=pd.cut(P.rul, list(edges)).astype(str))\n    t = d.groupby(\"band\", sort=False).agg(engines=(\"err\", \"size\"), rmse=(\"err\", lambda e: np.sqrt((e ** 2).mean())),\n                                          bias=(\"err\", \"mean\"))\n    t[\"share_of_sq_error_pct\"] = d.groupby(\"band\", sort=False).err.apply(lambda e: (e ** 2).sum()) / (d.err ** 2).sum() * 100\n    return t.loc[[b for b in d.sort_values(\"rul\").band.unique()]].round(1)\n\n\nband_table = error_by_band(fleet, CFG[\"rul_cap\"])\nprint(f\"test engines with true RUL > {CFG['rul_cap']}: {int((fleet.rul > CFG['rul_cap']).sum())} of {len(fleet)}\")\ndisplay(band_table)\n\n\ndef savefig(fig, name):\n    p = OUT / \"plots\" / f\"{name}.png\"\n    fig.savefig(p, dpi=120, bbox_inches=\"tight\")\n    plt.show()\n    return p\n\n\nplots = []\n\n# 1. every test engine: true vs predicted RUL with the 10-90% interval\nf = fleet.sort_values(\"rul\").reset_index(drop=True)\nfig, ax = plt.subplots(figsize=(12, 4))\nax.fill_between(f.index, f.rul_lo, f.rul_hi, alpha=0.25, label=\"10-90% interval\")\nax.plot(f.index, f.rul, \"k-\", lw=1.5, label=\"true RUL\")\nax.plot(f.index, f.rul_pred, \"o\", ms=3, label=\"predicted RUL\")\nax.set(xlabel=\"test engines (sorted by true RUL)\", ylabel=\"RUL (cycles)\",\n       title=f\"{SUBSET} test: RMSE {m_c['test_rmse_capped']:.1f} capped / {m_c['test_rmse']:.1f} true, \"\n             f\"NASA score {m_c['test_nasa_score']:.0f}\")\nax.axhline(CFG[\"rul_cap\"], color=\"grey\", ls=\":\", lw=1, label=f\"RUL cap {CFG['rul_cap']}\")\nax.legend()\nplots.append(savefig(fig, \"test_true_vs_pred\"))\n\n# 2. error distribution: positive = late (dangerous)\nerr = fleet.rul_pred - fleet.rul\nfig, axes = plt.subplots(1, 2, figsize=(12, 4))\naxes[0].scatter(fleet.rul, fleet.rul_pred, s=12)\nlim = [0, max(fleet.rul.max(), fleet.rul_pred.max()) + 5]\naxes[0].plot(lim, lim, \"k--\", lw=1)\naxes[0].set(xlabel=\"true RUL\", ylabel=\"predicted RUL\", title=\"predicted vs true\")\naxes[1].hist(err[err <= 0], bins=20, alpha=0.7, label=\"early (safe side)\")\naxes[1].hist(err[err > 0], bins=20, alpha=0.7, label=\"late (risk)\")\naxes[1].set(xlabel=\"predicted \u2212 true (cycles)\", ylabel=\"engines\", title=\"error distribution\")\naxes[1].legend()\nplots.append(savefig(fig, \"test_errors\"))\n\n# 3. validation engines over their whole life\nshow_units = sorted(Pv_c.unit.unique())[:4]\nfig, axes = plt.subplots(1, len(show_units), figsize=(16, 3.5), sharey=True)\nfor ax, u in zip(np.atleast_1d(axes), show_units):\n    e = Pv_c[Pv_c.unit == u]\n    ax.fill_between(e.cycle, e.rul_lo, e.rul_hi, alpha=0.25)\n    ax.plot(e.cycle, e.rul_capped, \"k-\", lw=1, label=\"true (capped)\")\n    ax.plot(e.cycle, e.rul_pred, lw=1.5, label=\"predicted\")\n    a = e[e.anomaly_persistent]\n    ax.plot(a.cycle, np.zeros(len(a)) + 2, \"|\", color=\"red\", ms=8, label=\"anomaly\")\n    ax.set(title=f\"val engine {u} \u00b7 champion v{CHAMPION_VERSION}\", xlabel=\"cycle\")\nnp.atleast_1d(axes)[0].set_ylabel(\"RUL\")\nnp.atleast_1d(axes)[0].legend(fontsize=8)\nplots.append(savefig(fig, \"val_trajectories\"))\n\n# 4. early-warning classifier precision/recall on validation engines\np_, r_, _ = precision_recall_curve(Pv_c.fail, Pv_c.p_fail)\nfig, ax = plt.subplots(figsize=(5, 4))\nax.plot(r_, p_)\nax.scatter([m_v[\"val_clf_recall\"]], [m_v[\"val_clf_precision\"]], color=\"red\", zorder=3, label=\"chosen threshold\")\nax.set(xlabel=\"recall\", ylabel=\"precision\", title=f\"P(fail \u2264 {CFG['fail_window']}) \u00b7 PR-AUC {m_v['val_clf_pr_auc']:.3f}\")\nax.legend()\nplots.append(savefig(fig, \"classifier_pr\"))\n\nwith tracker.run(f\"{SUBSET}_evaluation\", subset=SUBSET, stage=\"evaluation\", version=PRODUCTION_VERSION):\n    tracker.log_metrics(m_c)\n    for p in plots:\n        tracker.log_artifact(p)",
"AGENT_CFG = {\n    \"fail_window\": CFG[\"fail_window\"],\n    \"critical_rul_lower\": 15,\n    \"high_rul\": 30, \"high_p_fail\": 0.50,\n    \"medium_rul\": 60, \"low_rul\": 100,\n    \"review_interval_width\": 30,  # replaced by the production bundle's CV-derived review width when it exists\n    \"review_min_cycles\": 15,\n    \"conflict_p_fail\": 0.50, \"conflict_rul\": 80, \"conflict_low_p_fail\": 0.10,\n    \"safety_buffer\": 10,\n    \"deescalate_cycles\": 5,       # lower severity must hold this many consecutive cycles before an order is downgraded or closed\n    \"review_timeout_cycles\": 10,  # a NEEDS_REVIEW order not handled within this many cycles is flagged overdue (escalate to a supervisor)\n}\nSEVERITIES = [\"NORMAL\", \"LOW\", \"MEDIUM\", \"HIGH\", \"CRITICAL\"]\n# review reasons that mean the data behind an order can't be trusted (re-checked on already-open orders)\nDATA_TRUST_REASONS = (\"sensor fault\", \"operating condition not seen\", \"recent sensor data was imputed\")\nPRIORITY = {\"CRITICAL\": \"P1\", \"HIGH\": \"P2\", \"MEDIUM\": \"P3\"}\nACTIONS = {\n    \"CRITICAL\": \"Remove from service now; inspect and overhaul the {component} before the next operation.\",\n    \"HIGH\": \"Schedule {component} maintenance before the due cycle; reserve parts and a maintenance slot.\",\n    \"MEDIUM\": \"Plan an inspection of the {component} and increase monitoring frequency.\",\n}\n\n\nclass TriageAgent:\n    version = \"rules-1.0\"\n\n    def __init__(self, cfg, subset, model_version, fault_modes=1):\n        self.cfg, self.subset, self.model_version = cfg, subset, model_version\n        self.component = \"HPC module\" if fault_modes == 1 else \"HPC and fan modules\"\n        self.open_orders, self.work_orders, self.log = {}, [], []\n        self.calm = {}                                   # consecutive cycles below the open order's severity\n        self.sensor_tickets, self.open_sensor_tickets = [], {}\n\n    def assess(self, r):\n        c = self.cfg\n        rules = [\n            (\"CRITICAL\", r[\"rul_lo\"] <= c[\"critical_rul_lower\"], f\"lower-bound RUL {r['rul_lo']:.0f} <= {c['critical_rul_lower']}\"),\n            (\"HIGH\", r[\"rul_pred\"] <= c[\"high_rul\"], f\"predicted RUL {r['rul_pred']:.0f} <= {c['high_rul']}\"),\n            (\"HIGH\", r[\"p_fail\"] >= c[\"high_p_fail\"], f\"P(fail within {c['fail_window']}) {r['p_fail']:.2f} >= {c['high_p_fail']}\"),\n            (\"MEDIUM\", r[\"rul_pred\"] <= c[\"medium_rul\"], f\"predicted RUL {r['rul_pred']:.0f} <= {c['medium_rul']}\"),\n            # an anomaly raises MEDIUM only when the RUL model agrees the machine is past healthy (RUL <= low_rul);\n            # otherwise it is a LOW watch item (validation: same near-failure recall, 2-12x fewer orders on healthy cycles)\n            (\"MEDIUM\", bool(r[\"anomaly_persistent\"]) and r[\"rul_pred\"] <= c[\"low_rul\"],\n             f\"persistent anomaly (score {r['anomaly_score']:.3f}), RUL {r['rul_pred']:.0f} <= {c['low_rul']}\"),\n            (\"LOW\", bool(r[\"anomaly_persistent\"]) and r[\"rul_pred\"] > c[\"low_rul\"],\n             f\"persistent anomaly not confirmed by RUL model (watch)\"),\n            (\"LOW\", r[\"rul_pred\"] <= c[\"low_rul\"], f\"predicted RUL {r['rul_pred']:.0f} <= {c['low_rul']}\"),\n        ]\n        fired = [(sev, why) for sev, hit, why in rules if hit]\n        severity = max((s for s, _ in fired), key=SEVERITIES.index, default=\"NORMAL\")\n        review = []\n        if r[\"rul_hi\"] - r[\"rul_lo\"] > c[\"review_interval_width\"]:\n            review.append(f\"uncertain: interval {r['rul_lo']:.0f}-{r['rul_hi']:.0f} wider than {c['review_interval_width']}\")\n        if r[\"p_fail\"] >= c[\"conflict_p_fail\"] and r[\"rul_pred\"] > c[\"conflict_rul\"]:\n            review.append(\"models disagree: high failure probability but long predicted RUL\")\n        if r[\"rul_pred\"] <= c[\"high_rul\"] and r[\"p_fail\"] < c[\"conflict_low_p_fail\"]:\n            review.append(\"models disagree: short predicted RUL but low failure probability\")\n        if r[\"cycles_observed\"] < c[\"review_min_cycles\"]:\n            review.append(f\"only {int(r['cycles_observed'])} cycles of history\")\n        if bool(r[\"recent_imputed\"]):\n            review.append(\"recent sensor data was imputed\")\n        if bool(r.get(\"recent_unseen\", False)):\n            review.append(\"operating condition not seen in training (nearest known condition used)\")\n        if bool(r.get(\"sensor_fault\", False)):\n            review.append(f\"sensor fault: {r.get('sensor_fault_detail', '')}\")\n        return {\"severity\": severity, \"reasons\": [w for _, w in fired], \"review_reasons\": review}\n\n    def _work_order(self, r, d, supersedes):\n        sev = d[\"severity\"]\n        due = 0 if sev == \"CRITICAL\" else max(0, int(np.floor(r[\"rul_lo\"])) - self.cfg[\"safety_buffer\"])\n        status = \"OPEN\" if sev == \"CRITICAL\" or not d[\"review_reasons\"] else \"NEEDS_REVIEW\"\n        return {\n            \"work_order_id\": f\"WO-{self.subset}-{int(r['unit']):03d}-{int(r['cycle']):04d}\", \"type\": \"MAINTENANCE\",\n            \"created_at\": now_iso(), \"subset\": self.subset, \"unit\": int(r[\"unit\"]), \"at_cycle\": int(r[\"cycle\"]),\n            \"severity\": sev, \"priority\": PRIORITY[sev], \"status\": status,\n            \"human_review\": bool(d[\"review_reasons\"]),\n            \"predicted_rul\": round(float(r[\"rul_pred\"]), 1), \"rul_lower\": round(float(r[\"rul_lo\"]), 1),\n            \"rul_upper\": round(float(r[\"rul_hi\"]), 1), \"p_fail_window\": round(float(r[\"p_fail\"]), 3),\n            \"anomaly\": bool(r[\"anomaly_persistent\"]), \"due_within_cycles\": due,\n            \"recommended_action\": ACTIONS[sev].format(component=self.component),\n            \"reasons\": d[\"reasons\"], \"review_reasons\": d[\"review_reasons\"], \"supersedes\": supersedes,\n            \"model_version\": self.model_version, \"agent_version\": self.version,\n        }\n\n    def _sensor_ticket(self, r):\n        # a broken sensor is a data problem, not a machine problem: separate ticket per machine, re-raised only\n        # when another sensor joins the fault list\n        unit, detail = int(r[\"unit\"]), str(r.get(\"sensor_fault_detail\", \"\"))\n        faulty = {p.split(\" \")[0] for p in detail.split(\"; \") if p}\n        open_t = self.open_sensor_tickets.get(unit)\n        if open_t is not None and faulty <= set(open_t[\"sensors\"]):\n            return None\n        faulty |= set(open_t[\"sensors\"]) if open_t is not None else set()\n        t = {\"ticket_id\": f\"SQ-{self.subset}-{unit:03d}-{int(r['cycle']):04d}\", \"type\": \"CHECK_SENSOR\",\n             \"created_at\": now_iso(), \"subset\": self.subset, \"unit\": unit, \"at_cycle\": int(r[\"cycle\"]),\n             \"priority\": \"P3\", \"status\": \"OPEN\", \"sensors\": sorted(faulty), \"detail\": detail,\n             \"recommended_action\": f\"Inspect / replace sensor(s) {', '.join(sorted(faulty))} ({detail}). \"\n                                   \"Predictions for this machine are unreliable until fixed.\",\n             \"agent_version\": self.version}\n        if open_t is not None:\n            open_t[\"status\"] = \"SUPERSEDED\"\n        self.open_sensor_tickets[unit] = t\n        self.sensor_tickets.append(t)\n        return t[\"ticket_id\"]\n\n    def process(self, r):\n        d = self.assess(r)\n        sev, unit, cycle = d[\"severity\"], int(r[\"unit\"]), int(r[\"cycle\"])\n        rank = SEVERITIES.index\n        prev = self.open_orders.get(unit)\n        self.calm[unit] = self.calm.get(unit, 0) + 1 if prev is not None and rank(sev) < rank(prev[\"severity\"]) else 0\n        event, wo_id = \"none\", None\n        if sev in PRIORITY and (prev is None or rank(sev) > rank(prev[\"severity\"])):\n            if prev is not None:\n                prev[\"status\"] = \"SUPERSEDED\"\n            wo = self._work_order(r, d, prev[\"work_order_id\"] if prev else None)\n            self.open_orders[unit] = wo\n            self.work_orders.append(wo)\n            event, wo_id = (\"created\" if prev is None else \"escalated\"), wo[\"work_order_id\"]\n        elif prev is not None and self.calm[unit] >= self.cfg[\"deescalate_cycles\"]:\n            # hysteresis: only after K consecutive calmer cycles is the order downgraded, or closed if no order is needed\n            prev[\"closed_at_cycle\"] = cycle\n            if sev in PRIORITY:\n                prev[\"status\"] = \"SUPERSEDED\"\n                wo = self._work_order(r, d, prev[\"work_order_id\"])\n                self.open_orders[unit] = wo\n                self.work_orders.append(wo)\n                event, wo_id = \"downgraded\", wo[\"work_order_id\"]\n            else:\n                prev[\"status\"] = \"CLOSED_RECOVERED\"\n                del self.open_orders[unit]\n                event, wo_id = \"closed\", prev[\"work_order_id\"]\n            self.calm[unit] = 0\n        elif prev is not None:\n            event, wo_id = \"order_already_open\", prev[\"work_order_id\"]\n        elif sev == \"LOW\":\n            event = \"monitor\"\n        # data-trust problems that appear after an order was raised: the order rests on suspect data, so it goes\n        # to human review (CRITICAL stays dispatched, flagged). Uncertainty changes don't re-open review.\n        review_requested = False\n        cur = self.open_orders.get(unit)\n        trust = [x for x in d[\"review_reasons\"] if x.startswith(DATA_TRUST_REASONS)]\n        if cur is not None and event == \"order_already_open\" and trust:\n            new = [x for x in trust if x not in cur[\"review_reasons\"]]\n            if new:\n                cur[\"review_reasons\"] = cur[\"review_reasons\"] + new\n                cur[\"human_review\"] = True\n                cur.setdefault(\"review_requested_at_cycle\", cycle)\n                if cur[\"status\"] == \"OPEN\" and cur[\"severity\"] != \"CRITICAL\":\n                    cur[\"status\"] = \"NEEDS_REVIEW\"\n                review_requested = True\n        overdue = False\n        if (cur is not None and cur[\"status\"] == \"NEEDS_REVIEW\" and not cur.get(\"review_overdue\")\n                and cycle - cur[\"at_cycle\"] >= self.cfg[\"review_timeout_cycles\"]):\n            cur[\"review_overdue\"], cur[\"review_overdue_at_cycle\"], overdue = True, cycle, True\n        ticket = self._sensor_ticket(r) if bool(r.get(\"sensor_fault\", False)) else None\n        entry = {\"unit\": unit, \"cycle\": cycle, \"severity\": sev, \"event\": event, \"work_order_id\": wo_id,\n                 \"rul_pred\": float(r[\"rul_pred\"]), \"rul_lo\": float(r[\"rul_lo\"]), \"rul_hi\": float(r[\"rul_hi\"]),\n                 \"p_fail\": float(r[\"p_fail\"]), \"anomaly\": bool(r[\"anomaly_persistent\"]),\n                 \"human_review\": bool(d[\"review_reasons\"]), \"review_requested\": review_requested,\n                 \"review_overdue\": overdue, \"sensor_ticket\": ticket,\n                 \"reasons\": \"; \".join(d[\"reasons\"]), \"review_reasons\": \"; \".join(d[\"review_reasons\"])}\n        self.log.append(entry)\n        return entry\n\n    def triage(self, frame):\n        return pd.DataFrame([self.process(r) for _, r in frame.sort_values([\"unit\", \"cycle\"]).iterrows()])\n\n\ndef true_severity(rul, c=AGENT_CFG):\n    if rul <= c[\"critical_rul_lower\"]:\n        return \"CRITICAL\"\n    if rul <= c[\"high_rul\"]:\n        return \"HIGH\"\n    if rul <= c[\"medium_rul\"]:\n        return \"MEDIUM\"\n    if rul <= c[\"low_rul\"]:\n        return \"LOW\"\n    return \"NORMAL\"\n\n\ndef wilson_low(k, n, z=1.96):\n    # lower end of the 95% Wilson interval: what \"k of n caught\" supports with a small n\n    if n == 0:\n        return float(\"nan\")\n    p = k / n\n    return float((p + z * z / (2 * n) - z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / (1 + z * z / n))\n\n\ndef evaluate_agent(decisions, truth, c=AGENT_CFG):\n    d = decisions.merge(truth[[\"unit\", \"rul\"]], on=\"unit\")\n    d[\"true_severity\"] = d.rul.map(lambda x: true_severity(x, c))\n    has_order = d.severity.isin(list(PRIORITY))\n    high_plus = d.severity.isin([\"HIGH\", \"CRITICAL\"])\n    urgent = d.rul <= c[\"high_rul\"]\n    confusion = pd.crosstab(d.true_severity, d.severity).reindex(index=SEVERITIES, columns=SEVERITIES, fill_value=0)\n    summary = {\n        \"engines\": int(len(d)), \"work_orders\": int(has_order.sum()),\n        \"urgent_engines\": int(urgent.sum()),\n        \"urgent_high_or_critical\": int((urgent & high_plus).sum()),\n        \"urgent_high_or_critical_ci95_low\": wilson_low(int((urgent & high_plus).sum()), int(urgent.sum())),\n        \"urgent_caught\": int((urgent & has_order).sum()),\n        \"urgent_missed\": int((urgent & ~has_order).sum()),\n        \"false_alarms_true_rul_gt_low\": int(((d.rul > c[\"low_rul\"]) & has_order).sum()),\n        \"exact_severity_match_pct\": float((d.true_severity == d.severity).mean() * 100),\n        \"review_rate_pct\": float(d.loc[has_order, \"human_review\"].mean() * 100) if has_order.any() else 0.0,\n    }\n    return summary, confusion, d",
"AGENT_CFG[\"review_interval_width\"] = round(b_c[\"review_width\"], 1)\nprint(f\"review threshold: interval wider than {AGENT_CFG['review_interval_width']} cycles\")\n\n# the current fleet = every test engine at its latest observed cycle\nagent = TriageAgent(AGENT_CFG, SUBSET, model_version=f\"{REGISTRY_NAME}/v{PRODUCTION_VERSION}\",\n                    fault_modes=CFG[\"fault_modes\"])\ndecisions = agent.triage(fleet)\nwork_orders = agent.work_orders\nagent_summary, agent_confusion, agent_eval = evaluate_agent(decisions, fleet)\n\nprint(f\"{len(work_orders)} work orders from {len(fleet)} engines\")\ndisplay(decisions.severity.value_counts().reindex(SEVERITIES, fill_value=0).to_frame(\"engines\"))\ndisplay(pd.Series(agent_summary).to_frame(\"agent vs true RUL\"))\nprint(\"rows = true severity (from true RUL), columns = agent severity\")\ndisplay(agent_confusion)\nprint(\"example work order:\")\nprint(json.dumps(sorted(work_orders, key=lambda w: SEVERITIES.index(w[\"severity\"]), reverse=True)[0], indent=2))",
"def inject_faults(raw, prep, units):\n    raw = raw.copy()\n    sensors, plan = prep[\"sensors\"], []\n    hi, lo = pd.DataFrame(prep[\"sensor_range\"][\"hi\"]).T, pd.DataFrame(prep[\"sensor_range\"][\"lo\"]).T\n    last = lambda u, n: raw.index[raw.unit == u][-n:]\n    u_stuck, u_range, u_dead, u_unseen = units\n    s = next((x for x in [\"s11\", \"s4\", \"s7\"] if x in prep[\"stuck_sensors\"]), prep[\"stuck_sensors\"][0])\n    rows = last(u_stuck, 25)\n    raw.loc[rows, s] = raw.at[rows[0], s]\n    plan.append((u_stuck, f\"{s} stuck\", int(raw.at[rows[0], \"cycle\"])))\n    s = \"s4\" if \"s4\" in sensors else sensors[0]\n    rows = last(u_range, 5)\n    raw.loc[rows, s] = float((hi[s] + (hi[s] - lo[s])).max())\n    plan.append((u_range, f\"{s} out of range\", int(raw.at[rows[0], \"cycle\"])))\n    s = next((x for x in [\"s7\", \"s12\", \"s2\"] if x in prep[\"stuck_sensors\"]), prep[\"stuck_sensors\"][-1])\n    rows = last(u_dead, 20)\n    raw.loc[rows, s] = np.nan\n    plan.append((u_dead, f\"{s} dead\", int(raw.at[rows[0], \"cycle\"])))\n    rows = last(u_unseen, 10)\n    raw.loc[rows, OPS] = [30.0, 0.5, 80.0]\n    plan.append((u_unseen, \"unseen operating condition (30 / 0.5 / 80)\", int(raw.at[rows[0], \"cycle\"])))\n    return raw, plan\n\n\ndemo_units = [int(u) for u in fleet.sort_values(\"cycle\").unit.tail(4)]  # longest test histories\nraw_faulty, fault_plan = inject_faults(raw_test[raw_test.unit.isin(demo_units)], b_c[\"prep\"], demo_units)\nfaulty = add_labels(basic_clean(raw_faulty), CFG, rul_final[rul_final.unit.isin(demo_units)])\nP_faulty = predict_bundle(b_c, faulty)\ndemo_agent = TriageAgent(AGENT_CFG, SUBSET, model_version=f\"{REGISTRY_NAME}/v{PRODUCTION_VERSION}\",\n                         fault_modes=CFG[\"fault_modes\"])\ndemo_log = demo_agent.triage(P_faulty)\nclean_decision = decisions.set_index(\"unit\")\n\nrows = []\nfor u, what, start in fault_plan:\n    p, lg = P_faulty[P_faulty.unit == u], demo_log[demo_log.unit == u]\n    flagged = p[p.sensor_fault | p.unseen_condition]\n    gate = sorted({part for d in p.loc[p.sensor_fault, \"sensor_fault_detail\"] for part in d.split(\"; \") if part}\n                  | ({\"unseen operating condition\"} if p.unseen_condition.any() else set()))\n    order = demo_agent.open_orders.get(u)\n    rows.append({\"unit\": u, \"injected\": what, \"from_cycle\": start, \"detected\": bool(len(flagged)),\n                 \"first_flagged_cycle\": int(flagged.cycle.min()) if len(flagged) else None,\n                 \"gate_says\": \"; \".join(gate),\n                 \"check_sensor_tickets\": len([t for t in demo_agent.sensor_tickets if t[\"unit\"] == u]),\n                 \"severity_clean_data\": clean_decision.at[u, \"severity\"], \"severity_faulty_data\": lg.severity.iloc[-1],\n                 \"rul_clean_data\": round(float(clean_decision.at[u, \"rul_pred\"]), 1),\n                 \"rul_faulty_data\": round(float(lg.rul_pred.iloc[-1]), 1),\n                 \"order_status\": order[\"status\"] if order else \"no order\",\n                 \"order_flagged_for_review\": bool(order[\"human_review\"]) if order else False,\n                 \"review_reasons\": lg.review_reasons.iloc[-1]})\nfault_demo = pd.DataFrame(rows)\ndisplay(fault_demo)",
"def fleet_replay(P, model_version):\n    agent_ = TriageAgent(AGENT_CFG, SUBSET, model_version=model_version, fault_modes=CFG[\"fault_modes\"])\n    log = pd.DataFrame([agent_.process(r) for _, r in P.sort_values([\"unit\", \"cycle\"]).iterrows()])\n    return log.merge(P[[\"unit\", \"cycle\", \"rul\"]], on=[\"unit\", \"cycle\"]), agent_\n\n\ndef engine_stats(log, agent_, lives):\n    wo = pd.DataFrame(agent_.work_orders)\n    rows = []\n    for u, g in log.groupby(\"unit\"):\n        raised = g[g.event.isin([\"created\", \"escalated\", \"downgraded\"])]\n        first = lambda sevs: raised.loc[raised.severity.isin(sevs), \"rul\"].iloc[0] if raised.severity.isin(sevs).any() else np.nan\n        hc = wo[(wo.unit == u) & wo.severity.isin([\"HIGH\", \"CRITICAL\"])] if len(wo) else wo\n        service = int((hc.at_cycle + hc.due_within_cycles).min()) if len(hc) else None\n        life = int(lives[u])\n        failed = service is None or service >= life\n        rows.append({\"unit\": int(u), \"life\": life, \"first_order_rul\": first([\"MEDIUM\", \"HIGH\", \"CRITICAL\"]),\n                     \"first_high_rul\": first([\"HIGH\", \"CRITICAL\"]), \"first_critical_rul\": first([\"CRITICAL\"]),\n                     \"orders\": len(raised), \"downgrades_or_closures\": int(g.event.isin([\"downgraded\", \"closed\"]).sum()),\n                     \"reviews_overdue\": int(g.review_overdue.sum()), \"service_cycle\": service, \"failed\": failed,\n                     \"cycles_left_at_service\": None if failed else life - service})\n    return pd.DataFrame(rows)\n\n\ndef policy_eval(service, lives):\n    failed = [s is None or s >= l for s, l in zip(service, lives)]\n    used = [1.0 if f else s / l for s, l, f in zip(service, lives, failed)]\n    wasted = [0 if f else l - s for s, l, f in zip(service, lives, failed)]\n    return {\"unplanned_failures\": int(sum(failed)), \"engines\": len(lives),\n            \"life_used_pct\": float(np.mean(used) * 100), \"mean_cycles_wasted\": float(np.mean(wasted))}\n\n\nval_lives = Pv_c.groupby(\"unit\").cycle.max()\nreplay_log, replay_agent = fleet_replay(Pv_c, f\"{REGISTRY_NAME}/v{CHAMPION_VERSION}\")\nfleet_stats = engine_stats(replay_log, replay_agent, val_lives)\nfleet_replay_summary = {\n    \"engines\": int(len(fleet_stats)),\n    \"median_rul_at_first_order\": float(fleet_stats.first_order_rul.median()),\n    \"median_rul_at_first_high\": float(fleet_stats.first_high_rul.median()),\n    \"engines_critical_before_failure_pct\": float(fleet_stats.first_critical_rul.notna().mean() * 100),\n    \"mean_orders_per_engine\": float(fleet_stats.orders.mean()),\n    \"engines_with_downgrade_or_closure\": int((fleet_stats.downgrades_or_closures > 0).sum()),\n    \"reviews_overdue\": int(fleet_stats.reviews_overdue.sum()),\n    \"min_cycles_left_at_service\": float(fleet_stats.cycles_left_at_service.min()),\n    \"median_cycles_left_at_service\": float(fleet_stats.cycles_left_at_service.median()),\n}\n\nhist_lives = train_df[~train_df.unit.isin(SPLITS[\"val\"])].groupby(\"unit\").cycle.max()\nt_safe, t_5pct = int(hist_lives.min()) - 1, int(np.quantile(hist_lives, 0.05))\nlives = [int(val_lives[u]) for u in fleet_stats.unit]\npolicy_comparison = pd.DataFrame({\n    \"reactive (run to failure)\": policy_eval([None] * len(lives), lives),\n    f\"fixed interval T={t_safe} (0% failures in history)\": policy_eval([t_safe] * len(lives), lives),\n    f\"fixed interval T={t_5pct} (5% failures in history)\": policy_eval([t_5pct] * len(lives), lives),\n    \"predictive agent (ours)\": policy_eval([None if pd.isna(s) else int(s) for s in fleet_stats.service_cycle], lives),\n}).T\ndisplay(pd.Series(fleet_replay_summary).round(1).to_frame(\"fleet replay (validation engines)\"))\ndisplay(policy_comparison.round(1))\n\nreplay_unit = int(val_lives.idxmax())  # longest-lived validation engine, plotted below\nevents = replay_log[(replay_log.unit == replay_unit) & replay_log.event.isin([\"created\", \"escalated\", \"downgraded\", \"closed\"])]\ndisplay(events[[\"cycle\", \"rul\", \"severity\", \"event\", \"rul_pred\", \"rul_lo\", \"p_fail\", \"human_review\", \"reasons\"]]\n        .rename(columns={\"rul\": \"true_rul\"}))\n\ne = Pv_c[Pv_c.unit == replay_unit]\nfig, ax = plt.subplots(figsize=(12, 4))\nax.fill_between(e.cycle, e.rul_lo, e.rul_hi, alpha=0.2, label=\"10-90% interval\")\nax.plot(e.cycle, e.rul, \"k-\", lw=1, label=\"true RUL\")\nax.plot(e.cycle, e.rul_pred, lw=1.5, label=\"predicted RUL\")\ncolors = {\"MEDIUM\": \"gold\", \"HIGH\": \"orange\", \"CRITICAL\": \"red\"}\nfor _, ev in events.iterrows():\n    label = ev.severity if ev.event != \"closed\" else \"closed\"\n    ax.axvline(ev.cycle, color=colors.get(ev.severity, \"green\"), ls=\"--\", lw=1.2)\n    ax.text(ev.cycle, ax.get_ylim()[1] * 0.95, f\" {label}\\n true RUL {ev.rul}\", fontsize=8, va=\"top\")\nax.set(xlabel=\"cycle\", ylabel=\"RUL (cycles)\", title=f\"streaming replay \u00b7 validation engine {replay_unit}\")\nax.legend(loc=\"lower left\")\nplots.append(savefig(fig, \"replay\"))\n\n# today's practice vs ours: life used against unplanned failures\nfig, ax = plt.subplots(figsize=(7, 4))\ngrid = range(int(min(lives)) - 30, int(max(lives)) + 1, 5)\ncurve = [policy_eval([t] * len(lives), lives) for t in grid]\nax.plot([c[\"life_used_pct\"] for c in curve], [c[\"unplanned_failures\"] for c in curve], \"-\", color=\"grey\",\n        label=\"fixed interval (sweep of T)\")\nours = policy_comparison.loc[\"predictive agent (ours)\"]\nax.scatter([ours.life_used_pct], [ours.unplanned_failures], color=\"red\", zorder=3, s=60, label=\"predictive agent (ours)\")\nax.set(xlabel=\"useful life used before service (%)\", ylabel=f\"unplanned failures (of {len(lives)})\",\n       title=f\"{SUBSET}: maintenance policies on validation engines\")\nax.legend()\nplots.append(savefig(fig, \"policy_comparison\"))",
"shutil.copy(OUT / \"models\" / f\"bundle_{champion['bundle']['label']}.joblib\", OUT / \"champion_bundle.joblib\")\nshutil.copy(OUT / \"models\" / f\"bundle_{b_c['label']}.joblib\", OUT / \"production_bundle.joblib\")\nsave_json(b_c[\"prep\"], OUT / \"preprocessing.json\")\n\npred_cols = [\"unit\", \"cycle\", \"rul\", \"rul_pred\", \"rul_lo\", \"rul_hi\", \"p_fail\", \"fail_alert\", \"anomaly_score\",\n             \"anomaly_persistent\"]\nPv_c[pred_cols].to_csv(OUT / \"predictions_val.csv\", index=False)\nPt_c[pred_cols].to_csv(OUT / \"predictions_test.csv\", index=False)\nagent_eval.rename(columns={\"rul\": \"true_rul\"}).to_csv(OUT / \"fleet_status.csv\", index=False)\nsave_json(work_orders, OUT / \"work_orders.json\")\nwo_table = pd.DataFrame(work_orders)\nif len(wo_table):\n    wo_table[\"reasons\"] = wo_table.reasons.str.join(\"; \")\n    wo_table[\"review_reasons\"] = wo_table.review_reasons.str.join(\"; \")\nwo_table.to_csv(OUT / \"work_orders.csv\", index=False)\nreplay_log.to_csv(OUT / \"replay_log.csv\", index=False)\nfleet_stats.to_csv(OUT / \"fleet_engine_stats.csv\", index=False)\nsave_json(agent.sensor_tickets, OUT / \"sensor_tickets.json\")\nfault_demo.to_csv(OUT / \"fault_injection.csv\", index=False)\n\nmonitoring = {\n    \"subset\": SUBSET, \"seed\": SEED, \"run_tag\": RUN_TAG, \"generated_at\": now_iso(), \"registry_name\": REGISTRY_NAME,\n    \"champion_version\": CHAMPION_VERSION, \"champion_model\": champion[\"bundle\"][\"kind\"],\n    \"champion_metrics\": m_v, \"production_version\": PRODUCTION_VERSION, \"production_label\": b_c[\"label\"],\n    \"production_engines\": len(b_c[\"units\"]), \"production_metrics\": m_c,\n    \"refit_comparison\": refit_comparison.to_dict(), \"test_error_by_band\": band_table.to_dict(orient=\"index\"),\n    # selection/calibration of the production version; per-version details under \"selections\"\n    \"cv_results\": b_c[\"selection\"][\"cv_results\"].to_dict(orient=\"index\"), \"cv_classifier\": b_c[\"selection\"][\"cv_clf\"],\n    \"clf_threshold\": b_c[\"clf_threshold\"], \"cv_interval\": b_c[\"selection\"][\"cv_interval\"],\n    \"selections\": {s[\"label\"]: {\"engines\": s[\"engines\"], \"model\": s[\"kind\"], \"clf_threshold\": s[\"clf_threshold\"],\n                                \"cv_results\": s[\"cv_results\"].to_dict(orient=\"index\"), \"cv_classifier\": s[\"cv_clf\"],\n                                \"cv_interval\": s[\"cv_interval\"]}\n                   for s in [SEL_V1, globals().get(\"SEL_V2\")] if s is not None},\n    \"history\": history,\n    \"drift_psi\": drift.to_dict(), \"promotion\": promotion,\n    \"agent_summary\": agent_summary, \"agent_confusion\": agent_confusion.to_dict(),\n    \"fleet_replay\": fleet_replay_summary, \"policy_comparison\": policy_comparison.to_dict(orient=\"index\"),\n    \"policy_fixed_intervals\": {\"t_safe\": t_safe, \"t_5pct\": t_5pct},\n    \"fault_injection\": fault_demo.to_dict(orient=\"records\"),\n    \"config\": CFG, \"agent_config\": AGENT_CFG, \"splits\": SPLITS, \"ingest_report\": ingest_report,\n    \"sensors_kept\": b_c[\"prep\"][\"sensors\"], \"sensors_dropped\": b_c[\"prep\"][\"dropped\"],\n}\nsave_json(monitoring, OUT / \"monitoring.json\")\nsave_json({\"config\": CFG, \"agent_config\": AGENT_CFG}, OUT / \"config.json\")\n\nwith tracker.run(f\"{SUBSET}_agent\", subset=SUBSET, stage=\"agent\", version=PRODUCTION_VERSION):\n    tracker.log_params(AGENT_CFG)\n    tracker.log_metrics(prefix(\"agent\", agent_summary))\n    for name in [\"work_orders.json\", \"fleet_status.csv\", \"monitoring.json\"]:\n        tracker.log_artifact(OUT / name)\n\n# run_all_subsets.ipynb sets SKIP_ZIP and makes one combined zip at the end instead\nzip_path = \"skipped (combined zip made by run_all_subsets)\"\nif not globals().get(\"SKIP_ZIP\", False):\n    bundle_dir = WORK / f\"{SUBSET}_outputs\"\n    if bundle_dir.exists():\n        shutil.rmtree(bundle_dir)\n    shutil.copytree(OUT, bundle_dir / \"artifacts\")\n    shutil.copytree(WORK / \"tracking\", bundle_dir / \"tracking\")\n    zip_path = shutil.make_archive(str(bundle_dir), \"zip\", root_dir=bundle_dir)\n    shutil.rmtree(bundle_dir)\n\nprint(f\"champion: {REGISTRY_NAME} v{CHAMPION_VERSION} | production: v{PRODUCTION_VERSION} \"\n      f\"({b_c['kind']}, {len(b_c['units'])} engines)\")\nprint(f\"production test RMSE {m_c['test_rmse_capped']:.2f} capped / {m_c['test_rmse']:.2f} true | \"\n      f\"NASA score {m_c['test_nasa_score']:.1f} | \"\n      f\"urgent engines caught {agent_summary['urgent_caught']}/{agent_summary['urgent_engines']}\")\nprint(\"outputs:\", zip_path)"
]''')
# monitoring dashboard builder, embedded from scripts/build_dashboard.py
DASHBOARD_SRC = '"""Monitoring dashboard: one self-contained HTML page built from a run\'s saved outputs (no model loading, no server).\n\nReads artifacts/<SUBSET>/{monitoring.json, fleet_status.csv, work_orders.csv, predictions_test.csv, replay_log.csv,\nfleet_engine_stats.csv, fault_injection.csv} and artifacts/cost_optimization.json (optional), and writes\n<run>/dashboard.html. Opens offline in any browser; all charts are plain SVG.\n\n    python scripts/build_dashboard.py [run_dir]     # default: output/all_run_v3\n"""\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nSUBSETS = ["FD001", "FD002", "FD003", "FD004"]\nSEVERITIES = ["NORMAL", "LOW", "MEDIUM", "HIGH", "CRITICAL"]\nEVENTS = ["created", "escalated", "downgraded", "closed"]\n\n\ndef _r(a, d=1):\n    return [None if pd.isna(x) else round(float(x), d) for x in a]\n\n\ndef _clean(v):\n    # json.dumps can\'t write NaN/inf/numpy scalars\n    if isinstance(v, dict):\n        return {str(k): _clean(x) for k, x in v.items()}\n    if isinstance(v, (list, tuple)):\n        return [_clean(x) for x in v]\n    if isinstance(v, (np.integer,)):\n        return int(v)\n    if isinstance(v, (float, np.floating)):\n        return None if not np.isfinite(v) else round(float(v), 4)\n    if isinstance(v, np.bool_):\n        return bool(v)\n    return v\n\n\ndef _records(df, cols):\n    df = df[[c for c in cols if c in df.columns]].copy()\n    return [{k: (None if (not isinstance(v, str) and pd.isna(v)) else v) for k, v in row.items()}\n            for row in df.to_dict(orient="records")]\n\n\ndef subset_data(art):\n    mon = json.loads((art / "monitoring.json").read_text())\n    fleet = pd.read_csv(art / "fleet_status.csv")\n    orders = pd.read_csv(art / "work_orders.csv")\n    pred = pd.read_csv(art / "predictions_test.csv")\n    replay = pd.read_csv(art / "replay_log.csv")\n    stats = pd.read_csv(art / "fleet_engine_stats.csv")\n    tickets = json.loads((art / "sensor_tickets.json").read_text()) if (art / "sensor_tickets.json").exists() else []\n\n    test_traj = {}\n    for u, g in pred.sort_values(["unit", "cycle"]).groupby("unit"):\n        test_traj[int(u)] = {"c0": int(g.cycle.iloc[0]), "pred": _r(g.rul_pred), "lo": _r(g.rul_lo), "hi": _r(g.rul_hi),\n                             "p": _r(g.p_fail, 3), "true": [int(x) for x in g.rul]}\n\n    sev_ix = {s: i for i, s in enumerate(SEVERITIES)}\n    replay_traj, events = {}, []\n    for u, g in replay.sort_values(["unit", "cycle"]).groupby("unit"):\n        replay_traj[int(u)] = {"c0": int(g.cycle.iloc[0]), "pred": _r(g.rul_pred), "lo": _r(g.rul_lo),\n                               "hi": _r(g.rul_hi), "p": _r(g.p_fail, 3), "true": [int(x) for x in g.rul],\n                               "sev": [sev_ix.get(s, 0) for s in g.severity],\n                               "review": [int(bool(x)) for x in g.human_review]}\n        ev = g[g.event.isin(EVENTS)]\n        for row in ev.itertuples():\n            events.append({"unit": int(u), "cycle": int(row.cycle), "event": row.event, "severity": row.severity,\n                           "wo": row.work_order_id if isinstance(row.work_order_id, str) else "",\n                           "rul": round(float(row.rul_pred), 1), "true": int(row.rul),\n                           "why": row.reasons if isinstance(row.reasons, str) else ""})\n\n    fault = pd.read_csv(art / "fault_injection.csv") if (art / "fault_injection.csv").exists() else pd.DataFrame()\n    keep = ["subset", "generated_at", "registry_name", "champion_version", "champion_model", "production_version",\n            "production_label", "production_engines", "production_metrics", "test_error_by_band", "cv_results",\n            "cv_classifier", "cv_interval", "history", "drift_psi", "promotion", "agent_summary", "agent_confusion",\n            "fleet_replay", "policy_comparison", "policy_fixed_intervals", "config", "agent_config", "ingest_report",\n            "sensors_kept", "sensors_dropped", "selections"]\n    meta = {k: mon.get(k) for k in keep}\n    for s in (meta.get("selections") or {}).values():\n        s.pop("cv_results", None)\n    meta["history"] = [{k: h.get(k) for k in ["version", "label", "model", "trained_at", "train_engines",\n                                               "train_seconds", "test_rmse_capped", "test_rmse", "test_nasa_score",\n                                               "test_interval_coverage", "val_rmse_capped", "val_nasa_capped_mean"]}\n                       for h in mon.get("history", [])]\n    return _clean({\n        **meta,\n        "fleet": _records(fleet, ["unit", "cycle", "severity", "work_order_id", "rul_pred", "rul_lo", "rul_hi",\n                                  "p_fail", "anomaly", "human_review", "sensor_ticket", "reasons", "review_reasons",\n                                  "true_rul", "true_severity"]),\n        "orders": _records(orders, ["work_order_id", "type", "unit", "at_cycle", "severity", "priority", "status",\n                                    "human_review", "predicted_rul", "rul_lower", "rul_upper", "p_fail_window",\n                                    "anomaly", "due_within_cycles", "recommended_action", "reasons",\n                                    "review_reasons", "model_version", "agent_version"]),\n        "tickets": tickets,\n        "fault_injection": _records(fault, list(fault.columns)),\n        "engine_stats": _records(stats, list(stats.columns)),\n        "test_traj": test_traj,\n        "replay_traj": replay_traj,\n        "events": events,\n    })\n\n\ndef cost_data(run):\n    f = run / "artifacts" / "cost_optimization.json"\n    if not f.exists():\n        return None\n    t = json.loads(f.read_text())["table"]\n    return _clean([r for r in t if not r["policy"].startswith("agent, cost-tuned")])\n\n\ndef build(run, out=None):\n    run = Path(run)\n    data = {"subsets": {}, "cost": cost_data(run)}\n    for s in SUBSETS:\n        art = run / "artifacts" / s\n        if (art / "monitoring.json").exists():\n            data["subsets"][s] = subset_data(art)\n    if not data["subsets"]:\n        raise SystemExit(f"no artifacts/<subset>/monitoring.json under {run}")\n    payload = json.dumps(data, separators=(",", ":")).replace("</", "<\\\\/")\n    out = Path(out) if out else run / "dashboard.html"\n    out.write_text(TEMPLATE.replace("__DATA__", payload), encoding="utf-8")\n    return out\n\n\nTEMPLATE = r"""<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Fleet Monitor</title>\n<style>\n:root{color-scheme:light;--page:#f9f9f7;--surface:#fcfcfb;--raised:#ffffff;--ink:#0b0b0b;--ink2:#52514e;--muted:#898781;\n--grid:#e1e0d9;--axis:#c3c2b7;--ring:rgba(11,11,11,.10);--accent:#2a78d6;--accent-soft:#cde2fb;--band:rgba(42,120,214,.16);\n--good:#0ca30c;--warning:#fab219;--serious:#ec835a;--critical:#d03b3b;--low:#9a988f;--good-ink:#006300}\n@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){color-scheme:dark;--page:#0d0d0d;--surface:#1a1a19;--raised:#222221;\n--ink:#f0efec;--ink2:#c3c2b7;--grid:#2c2c2a;--axis:#383835;--ring:rgba(255,255,255,.10);--accent:#3987e5;--accent-soft:#184f95;\n--band:rgba(57,135,229,.24);--low:#6f6d66;--good-ink:#0ca30c}}\n:root[data-theme="dark"]{color-scheme:dark;--page:#0d0d0d;--surface:#1a1a19;--raised:#222221;--ink:#f0efec;--ink2:#c3c2b7;\n--grid:#2c2c2a;--axis:#383835;--ring:rgba(255,255,255,.10);--accent:#3987e5;--accent-soft:#184f95;--band:rgba(57,135,229,.24);\n--low:#6f6d66;--good-ink:#0ca30c}\n*{box-sizing:border-box}\nbody{margin:0;background:var(--page);color:var(--ink);font:14px/1.45 system-ui,-apple-system,"Segoe UI",sans-serif}\nheader{position:sticky;top:0;z-index:5;background:var(--surface);border-bottom:1px solid var(--ring);padding:10px 16px;\ndisplay:flex;flex-wrap:wrap;gap:10px 18px;align-items:center}\nh1{font-size:17px;margin:0;font-weight:650}\n.sub{color:var(--ink2);font-size:12.5px}\n.seg{display:inline-flex;border:1px solid var(--ring);border-radius:8px;overflow:hidden;flex-wrap:wrap}\n.seg button{border:0;background:transparent;color:var(--ink2);padding:6px 12px;font:inherit;cursor:pointer}\n.seg button[aria-pressed="true"]{background:var(--accent);color:#fff}\nnav.tabs{display:flex;gap:2px;flex-wrap:wrap;padding:0 16px;background:var(--surface);border-bottom:1px solid var(--ring)}\nnav.tabs button{border:0;background:none;padding:10px 12px;font:inherit;color:var(--ink2);cursor:pointer;border-bottom:2px solid transparent}\nnav.tabs button[aria-selected="true"]{color:var(--ink);border-bottom-color:var(--accent);font-weight:600}\nmain{padding:16px;max-width:1400px;margin:0 auto}\n.grid{display:grid;gap:14px}\n.g2{grid-template-columns:repeat(auto-fit,minmax(min(100%,460px),1fr))}\n.g3{grid-template-columns:repeat(auto-fit,minmax(min(100%,320px),1fr))}\n.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin-bottom:14px}\n.card{background:var(--surface);border:1px solid var(--ring);border-radius:10px;padding:14px;min-width:0}\n.card h2{font-size:14px;margin:0 0 2px;font-weight:620}\n.card .note{color:var(--muted);font-size:12px;margin:0 0 10px}\n.kpi .v{font-size:26px;font-weight:650;line-height:1.1}\n.kpi .l{color:var(--ink2);font-size:12.5px}\n.kpi .d{color:var(--muted);font-size:11.5px}\ntable{border-collapse:collapse;width:100%;font-size:13px}\nth,td{text-align:left;padding:6px 8px;border-bottom:1px solid var(--grid);vertical-align:top}\nth{color:var(--ink2);font-weight:600;font-size:12px;position:sticky;top:0;background:var(--surface)}\ntd.n,th.n{text-align:right;font-variant-numeric:tabular-nums}\ntbody tr.click{cursor:pointer}\ntbody tr.click:hover,tbody tr.sel{background:var(--accent-soft)}\n.scroll{max-height:520px;overflow:auto}\n.chip{display:inline-flex;align-items:center;gap:5px;padding:1px 8px;border-radius:99px;font-size:11.5px;font-weight:650;\nwhite-space:nowrap;color:#fff}\n.chip::before{content:"";width:6px;height:6px;border-radius:50%;background:currentColor;opacity:.85}\n.s-CRITICAL{background:var(--critical)}.s-HIGH{background:var(--serious)}.s-MEDIUM{background:var(--warning);color:#0b0b0b}\n.s-LOW{background:var(--low)}.s-NORMAL{background:var(--good)}\n.tag{display:inline-block;border:1px solid var(--ring);border-radius:6px;padding:0 6px;font-size:11.5px;color:var(--ink2);margin:1px 2px 1px 0}\n.tag.warn{border-color:var(--serious);color:var(--ink)}\n.filters{display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin-bottom:10px}\n.filters label{font-size:12.5px;color:var(--ink2);display:inline-flex;gap:5px;align-items:center}\nselect,input[type=search]{font:inherit;padding:4px 8px;border:1px solid var(--ring);border-radius:6px;background:var(--raised);color:var(--ink)}\n.wo{font-family:ui-monospace,Consolas,monospace;font-size:12.5px;background:var(--page);border:1px solid var(--ring);\nborder-radius:8px;padding:10px;white-space:pre-wrap}\n.ok{color:var(--good-ink);font-weight:600}.bad{color:var(--critical);font-weight:600}\n.legend{display:flex;gap:12px;flex-wrap:wrap;font-size:12px;color:var(--ink2);margin-top:6px}\n.legend i{display:inline-block;width:14px;height:3px;border-radius:2px;vertical-align:middle;margin-right:4px}\nsvg text{fill:var(--muted);font-size:11px}\n.tip{position:fixed;pointer-events:none;z-index:20;background:var(--raised);color:var(--ink);border:1px solid var(--ring);\nborder-radius:8px;padding:6px 9px;font-size:12px;box-shadow:0 4px 14px rgba(0,0,0,.14);display:none;max-width:280px}\n.heat td{text-align:center;font-variant-numeric:tabular-nums;border:2px solid var(--surface);border-radius:4px}\n.strip{display:grid;grid-template-columns:repeat(auto-fill,minmax(54px,1fr));gap:4px}\n.cell{border-radius:6px;padding:4px 2px;text-align:center;font-size:11px;font-weight:650;color:#fff;cursor:pointer;border:2px solid transparent}\n.cell.done{background:var(--grid)!important;color:var(--muted)}\n.cell.rv{border-color:var(--ink)}\n.feed{max-height:330px;overflow:auto;font-size:12.5px}\n.feed div{padding:4px 0;border-bottom:1px solid var(--grid)}\n.play{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-bottom:10px}\n.play button{font:inherit;padding:5px 14px;border-radius:6px;border:1px solid var(--ring);background:var(--accent);color:#fff;cursor:pointer}\ninput[type=range]{flex:1;min-width:160px;accent-color:var(--accent)}\n.muted{color:var(--muted)}\n.two{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,520px),1fr));gap:14px}\n@media (max-width:600px){header{padding:10px 16px}.kpi .v{font-size:22px}}\n</style></head>\n<body>\n<header>\n  <div><h1>Fleet Monitor</h1><div class="sub" id="runinfo"></div></div>\n  <div class="seg" id="subsetSeg" role="group" aria-label="Subset"></div>\n  <div class="sub" id="modelinfo" style="margin-left:auto"></div>\n</header>\n<nav class="tabs" role="tablist" id="tabs"></nav>\n<main id="view"></main>\n<div class="tip" id="tip"></div>\n<script>\nconst DATA = __DATA__;\nconst SEV = ["NORMAL","LOW","MEDIUM","HIGH","CRITICAL"];\nconst SEVVAR = {NORMAL:"--good",LOW:"--low",MEDIUM:"--warning",HIGH:"--serious",CRITICAL:"--critical"};\nconst TABS = [["overview","Overview"],["fleet","Fleet now"],["orders","Work orders"],["replay","Live replay"],\n              ["mlops","Model & MLOps"],["quality","Data quality"],["impact","Agent & impact"]];\nconst subsets = Object.keys(DATA.subsets);\nlet state = {subset: subsets[0], tab: "overview", unit: null, sevFilter: "all", reviewOnly: false, status: "all", q: ""};\ntry { const s = JSON.parse(localStorage.getItem("fleetmon") || "{}"); if (s.tab) state.tab = s.tab; if (subsets.includes(s.subset)) state.subset = s.subset; } catch(e) {}\nconst $ = id => document.getElementById(id);\nconst css = v => getComputedStyle(document.documentElement).getPropertyValue(v).trim();\nconst esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",\'"\':"&quot;"}[c]));\nconst f1 = x => x == null ? "–" : (+x).toFixed(1);\nconst f0 = x => x == null ? "–" : Math.round(+x).toString();\nconst pct = (x, d=0) => x == null ? "–" : (100 * x).toFixed(d) + "%";\nconst chip = s => `<span class="chip s-${s}">${s}</span>`;\nconst D = () => DATA.subsets[state.subset];\nconst split = s => (s || "").split(";").map(x => x.trim()).filter(Boolean);\n\n// ---------- tooltip ----------\nconst tip = $("tip");\nfunction showTip(e, html) { tip.innerHTML = html; tip.style.display = "block";\n  const x = Math.min(e.clientX + 14, innerWidth - tip.offsetWidth - 8), y = Math.min(e.clientY + 14, innerHeight - tip.offsetHeight - 8);\n  tip.style.left = x + "px"; tip.style.top = y + "px"; }\nfunction hideTip() { tip.style.display = "none"; }\ndocument.addEventListener("mousemove", e => { const t = e.target.closest("[data-tip]"); if (t) showTip(e, t.dataset.tip); else hideTip(); });\n\n// ---------- SVG helpers ----------\nfunction scale(d0, d1, r0, r1) { const k = (r1 - r0) / ((d1 - d0) || 1); return v => r0 + (v - d0) * k; }\nfunction ticks(lo, hi, n=5) { const step = Math.pow(10, Math.floor(Math.log10((hi - lo) / n || 1))); const m = [1,2,5,10].find(m => (hi - lo) / (step * m) <= n) * step;\n  const out = []; for (let v = Math.ceil(lo / m) * m; v <= hi + 1e-9; v += m) out.push(+v.toFixed(6)); return out; }\n\n// horizontal bars: rows [{label, value, tip, color, ref}], one measure\nfunction hbars(rows, {fmt = f1, max = null, refLine = null, refLabel = "", w = 520, color = "--accent"} = {}) {\n  const rowH = 26, left = 150, right = 56, top = refLine != null ? 16 : 4, h = top + rows.length * rowH + 22;\n  const mx = max ?? (Math.max(...rows.map(r => r.value), refLine ?? 0) * 1.05 || 1);\n  const x = scale(0, mx, left, w - right);\n  let s = `<svg viewBox="0 0 ${w} ${h}" width="100%" role="img">`;\n  for (const t of ticks(0, mx, 4)) s += `<line x1="${x(t)}" x2="${x(t)}" y1="${top}" y2="${h - 20}" stroke="${css("--grid")}"/><text x="${x(t)}" y="${h - 6}" text-anchor="middle">${fmt(t)}</text>`;\n  rows.forEach((r, i) => { const y = top + i * rowH + 5, bw = Math.max(0, x(r.value) - left);\n    s += `<text x="${left - 8}" y="${y + 12}" text-anchor="end" style="fill:${css("--ink2")}">${esc(r.label)}</text>`;\n    s += `<g data-tip="${esc(r.tip || r.label + ": " + fmt(r.value))}"><rect x="${left}" y="${y - 3}" width="${w - right - left}" height="${rowH - 4}" fill="transparent"/>`;\n    s += `<path d="M${left},${y} h${Math.max(0, bw - 4)} a4,4 0 0 1 4,4 v8 a4,4 0 0 1 -4,4 h-${Math.max(0, bw - 4)} z" fill="${css(r.color || color)}"/></g>`;\n    s += `<text x="${left + bw + 6}" y="${y + 12}" style="fill:${css("--ink")}">${fmt(r.value)}</text>`; });\n  if (refLine != null) s += `<line x1="${x(refLine)}" x2="${x(refLine)}" y1="${top - 4}" y2="${h - 20}" stroke="${css("--critical")}" stroke-dasharray="4 3" stroke-width="1.5"/><text x="${x(refLine)}" y="${top - 6}" text-anchor="middle" style="fill:${css("--ink2")}">${esc(refLabel)}</text>`;\n  return s + "</svg>";\n}\n\n// RUL trajectory: predicted line + 80% band + actual (dashed); optional severity strip and cursor\nfunction trajChart(t, {cursor = null, h = 260, w = 640} = {}) {\n  const n = t.pred.length, c = i => t.c0 + i;\n  const L = 40, R = 12, T = 10, B = t.sev ? 40 : 26, ph = h - T - B;\n  const ymax = Math.max(130, ...t.true.slice(0, n).map(v => Math.min(v, 200)));\n  const x = scale(t.c0, t.c0 + n - 1, L, w - R), y = scale(0, ymax, T + ph, T);\n  let s = `<svg viewBox="0 0 ${w} ${h}" width="100%" role="img" aria-label="Predicted remaining useful life over cycles">`;\n  for (const v of ticks(0, ymax, 5)) s += `<line x1="${L}" x2="${w - R}" y1="${y(v)}" y2="${y(v)}" stroke="${css("--grid")}"/><text x="${L - 6}" y="${y(v) + 4}" text-anchor="end">${v}</text>`;\n  for (const v of ticks(t.c0, t.c0 + n - 1, 6)) s += `<text x="${x(v)}" y="${T + ph + 14}" text-anchor="middle">${v}</text>`;\n  s += `<line x1="${L}" x2="${w - R}" y1="${y(0)}" y2="${y(0)}" stroke="${css("--axis")}"/>`;\n  [15, 30].forEach(v => s += `<line x1="${L}" x2="${w - R}" y1="${y(v)}" y2="${y(v)}" stroke="${css(v === 15 ? "--critical" : "--serious")}" stroke-dasharray="3 4" opacity=".7"/>`);\n  let band = ""; for (let i = 0; i < n; i++) band += `${i ? "L" : "M"}${x(c(i)).toFixed(1)},${y(t.hi[i]).toFixed(1)}`;\n  for (let i = n - 1; i >= 0; i--) band += `L${x(c(i)).toFixed(1)},${y(t.lo[i]).toFixed(1)}`;\n  s += `<path d="${band}Z" fill="${css("--band")}"/>`;\n  const line = arr => arr.map((v, i) => `${i ? "L" : "M"}${x(c(i)).toFixed(1)},${y(Math.min(v, ymax)).toFixed(1)}`).join("");\n  s += `<path d="${line(t.true)}" fill="none" stroke="${css("--ink2")}" stroke-width="1.5" stroke-dasharray="5 4"/>`;\n  s += `<path d="${line(t.pred)}" fill="none" stroke="${css("--accent")}" stroke-width="2"/>`;\n  if (t.sev) { const sw = (w - R - L) / n;\n    for (let i = 0; i < n; i++) s += `<rect x="${(x(c(i)) - sw / 2).toFixed(1)}" y="${T + ph + 20}" width="${(sw + .6).toFixed(1)}" height="8" fill="${css(SEVVAR[SEV[t.sev[i]]])}"/>`;\n    s += `<text x="${L - 6}" y="${T + ph + 28}" text-anchor="end">agent</text>`; }\n  if (cursor != null && cursor >= t.c0 && cursor < t.c0 + n) s += `<line x1="${x(cursor)}" x2="${x(cursor)}" y1="${T}" y2="${T + ph}" stroke="${css("--ink")}" stroke-width="1"/>`;\n  s += `<line class="hx" x1="0" x2="0" y1="${T}" y2="${T + ph}" stroke="${css("--muted")}" visibility="hidden"/>`;\n  s += `<rect class="hit" x="${L}" y="${T}" width="${w - L - R}" height="${ph}" fill="transparent" data-c0="${t.c0}" data-n="${n}" data-l="${L}" data-w="${w - L - R}"/>`;\n  return s + "</svg>";\n}\nfunction bindTraj(el, t) {\n  const svg = el.querySelector("svg"); if (!svg) return; const hit = svg.querySelector(".hit"), hx = svg.querySelector(".hx");\n  hit.addEventListener("mousemove", e => { const r = hit.getBoundingClientRect(); const n = +hit.dataset.n;\n    const i = Math.max(0, Math.min(n - 1, Math.round((e.clientX - r.left) / r.width * (n - 1))));\n    const vb = svg.viewBox.baseVal, px = +hit.dataset.l + i / Math.max(1, n - 1) * +hit.dataset.w;\n    hx.setAttribute("x1", px); hx.setAttribute("x2", px); hx.setAttribute("visibility", "visible");\n    showTip(e, `<b>cycle ${t.c0 + i}</b><br>predicted RUL ${f1(t.pred[i])} <span class="muted">(80% range ${f0(t.lo[i])}–${f0(t.hi[i])})</span><br>actual RUL ${t.true[i]}<br>P(fail ≤ 30) ${(+t.p[i]).toFixed(3)}${t.sev ? "<br>agent: " + SEV[t.sev[i]] + (t.review && t.review[i] ? " · review" : "") : ""}`);\n    e.stopPropagation(); });\n  hit.addEventListener("mouseleave", () => { hx.setAttribute("visibility", "hidden"); hideTip(); });\n}\nconst trajLegend = sev => `<div class="legend"><span><i style="background:var(--accent)"></i>predicted RUL</span><span><i style="background:var(--band);height:10px"></i>80% range</span><span><i style="background:var(--ink2)"></i>actual RUL (dashed, known only in evaluation)</span><span><i style="background:var(--critical)"></i>15</span><span><i style="background:var(--serious)"></i>30 cycles</span>${sev ? SEV.map(s => `<span><i style="background:var(${SEVVAR[s]});height:8px"></i>${s}</span>`).join("") : ""}</div>`;\n\n// ---------- header / nav ----------\nfunction renderHeader() {\n  $("subsetSeg").innerHTML = subsets.map(s => `<button aria-pressed="${s === state.subset}" data-s="${s}">${s}</button>`).join("");\n  $("subsetSeg").onclick = e => { const b = e.target.closest("button"); if (!b) return; state.subset = b.dataset.s; state.unit = null; render(); };\n  const d = D();\n  $("runinfo").textContent = `Run ${String(d.generated_at || "").replace("T", " ").slice(0, 16)} UTC · NASA C-MAPSS`;\n  $("modelinfo").innerHTML = `Production: <b>${esc(d.registry_name)} v${d.production_version}</b> (${esc(d.production_label)}) · champion v${d.champion_version} · ${esc(d.champion_model)}`;\n  $("tabs").innerHTML = TABS.map(([k, l]) => `<button role="tab" aria-selected="${k === state.tab}" data-t="${k}">${l}</button>`).join("");\n  $("tabs").onclick = e => { const b = e.target.closest("button"); if (!b) return; state.tab = b.dataset.t; render(); };\n}\nfunction render() {\n  try { localStorage.setItem("fleetmon", JSON.stringify({tab: state.tab, subset: state.subset})); } catch(e) {}\n  stopPlay(); renderHeader(); $("view").innerHTML = ""; VIEWS[state.tab]();\n}\nconst kpi = (v, l, d = "") => `<div class="card kpi"><div class="v">${v}</div><div class="l">${l}</div>${d ? `<div class="d">${d}</div>` : ""}</div>`;\n\n// ---------- views ----------\nconst VIEWS = {};\nVIEWS.overview = () => {\n  const rows = subsets.map(s => { const d = DATA.subsets[s], m = d.production_metrics || {}, a = d.agent_summary || {}, pc = d.policy_comparison || {};\n    const ours = Object.entries(pc).find(([k]) => k.startsWith("predictive")), safe = Object.entries(pc).find(([k]) => k.includes("0% failures"));\n    const open = d.orders.filter(o => o.status === "OPEN").length, rev = d.orders.filter(o => o.status === "NEEDS_REVIEW").length;\n    const crit = d.fleet.filter(f => f.severity === "CRITICAL").length;\n    return {s, d, m, a, ours: ours && ours[1], safe: safe && safe[1], open, rev, crit}; });\n  const tot = k => rows.reduce((a, r) => a + r[k], 0);\n  const engines = rows.reduce((a, r) => a + r.d.fleet.length, 0);\n  const urgent = rows.reduce((a, r) => a + (r.a.urgent_high_or_critical || 0), 0), urgentN = rows.reduce((a, r) => a + (r.a.urgent_engines || 0), 0);\n  const fa = rows.reduce((a, r) => a + (r.a.false_alarms_true_rul_gt_low || 0), 0);\n  $("view").innerHTML = `<div class="kpis">\n    ${kpi(engines, "engines monitored", "test fleets, all four subsets")}\n    ${kpi(tot("open"), "work orders open", "created automatically")}\n    ${kpi(tot("rev"), "need human review", "exceptions only")}\n    ${kpi(tot("crit"), "CRITICAL now", "remove from service")}\n    ${kpi(`${urgent}/${urgentN}`, "near-failure engines at HIGH+", "true RUL ≤ 30")}\n    ${kpi(fa, "false alarms", "HIGH+ with true RUL > 100")}</div>\n  <div class="card"><h2>All subsets at a glance</h2><p class="note">Production model per subset, scored on the held-out NASA test engines. Click a row to open that subset.</p>\n  <div class="scroll"><table><thead><tr><th>Subset</th><th>Conditions · faults</th><th>Model</th><th class="n">RMSE (capped)</th><th class="n">RMSE (true)</th><th class="n">80% range coverage</th>\n  <th class="n">Near-failure at HIGH+</th><th class="n">False alarms</th><th class="n">Open · review</th><th class="n">Life used: agent vs fixed</th></tr></thead><tbody>\n  ${rows.map(r => `<tr class="click" data-s="${r.s}"><td><b>${r.s}</b></td><td>${r.d.config.condition_norm ? 6 : 1} · ${r.d.config.fault_modes}</td><td>${esc(r.d.champion_model)} v${r.d.production_version}</td>\n  <td class="n">${f1(r.m.test_rmse_capped)}</td><td class="n">${f1(r.m.test_rmse)}</td><td class="n">${pct(r.m.test_interval_coverage)}</td>\n  <td class="n">${r.a.urgent_high_or_critical}/${r.a.urgent_engines}</td><td class="n">${r.a.false_alarms_true_rul_gt_low}</td><td class="n">${r.open} · ${r.rev}</td>\n  <td class="n">${r.ours ? f0(r.ours.life_used_pct) + "%" : "–"} vs ${r.safe ? f0(r.safe.life_used_pct) + "%" : "–"}</td></tr>`).join("")}</tbody></table></div></div>\n  <div class="grid g2" style="margin-top:14px">\n   <div class="card"><h2>Fleet severity now, by subset</h2><p class="note">Agent severity at each test engine\'s latest cycle.</p><div id="ovSev"></div></div>\n   <div class="card"><h2>What happens to each engine</h2><p class="note">Replay of the validation engines to failure (not used in training). Fixed schedule = longest interval with no failures in past history.</p><div id="ovPol"></div></div></div>`;\n  $("view").querySelector("tbody").onclick = e => { const tr = e.target.closest("tr"); if (tr) { state.subset = tr.dataset.s; state.tab = "fleet"; render(); } };\n  $("ovSev").innerHTML = rows.map(r => stackRow(r.s, r.d.fleet)).join("") + sevLegend();\n  $("ovPol").innerHTML = hbars(rows.flatMap(r => [\n    {label: `${r.s} agent`, value: r.ours ? r.ours.life_used_pct : 0, tip: `${r.s} predictive agent: ${f1(r.ours?.life_used_pct)}% of life used, ${r.ours?.unplanned_failures} unplanned failures`},\n    {label: `${r.s} fixed`, value: r.safe ? r.safe.life_used_pct : 0, color: "--low", tip: `${r.s} fixed interval: ${f1(r.safe?.life_used_pct)}% of life used, ${r.safe?.unplanned_failures} unplanned failures`}]),\n    {max: 100, fmt: v => f0(v) + "%"}) + `<div class="legend"><span><i style="background:var(--accent);height:8px"></i>predictive agent</span><span><i style="background:var(--low);height:8px"></i>fixed interval</span><span>both: 0 unplanned failures</span></div>`;\n};\nfunction stackRow(label, fleet) {\n  const n = fleet.length; let x = 0, s = `<div style="display:flex;align-items:center;gap:8px;margin:6px 0"><div style="width:52px;font-size:12px;color:var(--ink2)">${label}</div><div style="flex:1;display:flex;gap:2px;height:18px">`;\n  for (const sv of SEV) { const k = fleet.filter(f => f.severity === sv).length; if (!k) continue;\n    s += `<div data-tip="${label} · ${sv}: ${k} of ${n} engines" style="flex:${k};background:var(${SEVVAR[sv]});border-radius:4px;min-width:3px"></div>`; }\n  return s + `</div><div style="width:34px;font-size:12px;text-align:right" class="muted">${n}</div></div>`;\n}\nconst sevLegend = () => `<div class="legend">${SEV.map(s => `<span><i style="background:var(${SEVVAR[s]});height:8px"></i>${s}</span>`).join("")}</div>`;\n\nVIEWS.fleet = () => {\n  const d = D(), fl = d.fleet;\n  const cnt = s => fl.filter(f => f.severity === s).length;\n  $("view").innerHTML = `<div class="kpis">${kpi(fl.length, "engines monitored", "test fleet, latest cycle")}${kpi(cnt("CRITICAL"), "CRITICAL", "remove from service now")}\n    ${kpi(cnt("HIGH"), "HIGH", "schedule before due date")}${kpi(cnt("MEDIUM"), "MEDIUM", "inspect")}${kpi(fl.filter(f => f.human_review).length, "flagged for review", "exceptions for a human")}\n    ${kpi(fl.filter(f => f.sensor_ticket).length, "sensor tickets", "stuck / dead / out-of-range sensor")}</div>\n  <div class="card" style="margin-bottom:14px"><h2>Severity mix</h2>${stackRow(state.subset, fl)}${sevLegend()}</div>\n  <div class="two"><div class="card"><h2>Fleet status</h2><p class="note">One row per engine, latest cycle. Click an engine for its history.</p>\n   <div class="filters"><label>Severity <select id="fSev"><option value="all">all</option>${SEV.slice().reverse().map(s => `<option ${state.sevFilter === s ? "selected" : ""}>${s}</option>`).join("")}</select></label>\n   <label><input type="checkbox" id="fRev" ${state.reviewOnly ? "checked" : ""}> review only</label><span class="muted" id="fCount"></span></div>\n   <div class="scroll"><table><thead><tr><th class="n">Engine</th><th class="n">Cycle</th><th>Severity</th><th class="n">RUL</th><th>80% range</th><th class="n">P(fail)</th><th>Flags</th><th class="n">Actual</th></tr></thead><tbody id="fBody"></tbody></table></div></div>\n   <div class="card" id="engine"></div></div>`;\n  const body = () => { const rows = fl.filter(f => (state.sevFilter === "all" || f.severity === state.sevFilter) && (!state.reviewOnly || f.human_review))\n      .sort((a, b) => SEV.indexOf(b.severity) - SEV.indexOf(a.severity) || a.rul_pred - b.rul_pred);\n    $("fCount").textContent = `${rows.length} engines`;\n    $("fBody").innerHTML = rows.map(f => `<tr class="click ${f.unit === state.unit ? "sel" : ""}" data-u="${f.unit}"><td class="n">${f.unit}</td><td class="n">${f.cycle}</td><td>${chip(f.severity)}</td>\n      <td class="n">${f0(f.rul_pred)}</td><td>${rangeBar(f.rul_lo, f.rul_pred, f.rul_hi)}</td><td class="n">${(+f.p_fail).toFixed(2)}</td>\n      <td>${f.human_review ? `<span class="tag warn" data-tip="${esc(f.review_reasons)}">review</span>` : ""}${f.anomaly ? \'<span class="tag">anomaly</span>\' : ""}${f.sensor_ticket ? \'<span class="tag warn">sensor</span>\' : ""}</td>\n      <td class="n muted">${f.true_rul}</td></tr>`).join(""); };\n  body();\n  $("fSev").onchange = e => { state.sevFilter = e.target.value; body(); };\n  $("fRev").onchange = e => { state.reviewOnly = e.target.checked; body(); };\n  $("fBody").onclick = e => { const tr = e.target.closest("tr"); if (!tr) return; state.unit = +tr.dataset.u; body(); engineView(); };\n  if (state.unit == null || !fl.some(f => f.unit === state.unit)) state.unit = fl.slice().sort((a, b) => SEV.indexOf(b.severity) - SEV.indexOf(a.severity) || a.rul_pred - b.rul_pred)[0].unit;\n  body(); engineView();\n};\nfunction rangeBar(lo, p, hi) { const x = v => Math.max(0, Math.min(100, v / 130 * 100));\n  return `<div data-tip="80% range ${f0(lo)}–${f0(hi)} cycles, predicted ${f0(p)}" style="position:relative;width:110px;height:12px;margin-top:3px;background:var(--grid);border-radius:3px">\n  <div style="position:absolute;left:${x(lo)}%;width:${Math.max(1, x(hi) - x(lo))}%;top:2px;height:8px;background:var(--band);border-radius:3px"></div>\n  <div style="position:absolute;left:calc(${x(p)}% - 1px);width:3px;top:0;height:12px;background:var(--accent);border-radius:2px"></div></div>`; }\nfunction engineView() {\n  const d = D(), u = state.unit, f = d.fleet.find(x => x.unit === u), t = d.test_traj[u];\n  const os = d.orders.filter(o => o.unit === u);\n  $("engine").innerHTML = `<h2>Engine ${u} ${chip(f.severity)}</h2><p class="note">Cycle ${f.cycle} · actual RUL ${f.true_rul} (true severity ${f.true_severity}) · the actual line is test ground truth, never seen by the model.</p>\n   <div id="engChart">${t ? trajChart(t) : ""}</div>${trajLegend(false)}\n   <h2 style="margin-top:14px">Why</h2><div>${split(f.reasons).map(r => `<span class="tag">${esc(r)}</span>`).join("") || \'<span class="muted">no rule fired</span>\'}</div>\n   ${f.review_reasons ? `<div style="margin-top:6px">${split(f.review_reasons).map(r => `<span class="tag warn">${esc(r)}</span>`).join("")}</div>` : ""}\n   <h2 style="margin-top:14px">Work order</h2>${os.length ? os.map(woCard).join("") : \'<p class="muted">No order: the agent logs and watches this engine.</p>\'}`;\n  if (t) bindTraj($("engChart"), t);\n}\nfunction woCard(o) {\n  return `<div class="wo">${esc(o.work_order_id)} · ${o.type} · ${o.severity} · ${o.priority} · ${o.status}\nEngine ${o.unit} · cycle ${o.at_cycle} · due within ${o.due_within_cycles} cycles\nRUL ${f1(o.predicted_rul)} (range ${f0(o.rul_lower)}–${f0(o.rul_upper)}) · P(fail ≤ 30) = ${(+o.p_fail_window).toFixed(3)}\nAction: ${esc(o.recommended_action)}\nWhy: ${esc(o.reasons)}${o.review_reasons ? "\\nReview: " + esc(o.review_reasons) : ""}\nModel ${esc(o.model_version)} · agent ${esc(o.agent_version)}</div>`;\n}\n\nVIEWS.orders = () => {\n  const d = D(), os = d.orders;\n  const by = s => os.filter(o => o.severity === s).length;\n  $("view").innerHTML = `<div class="kpis">${kpi(os.length, "work orders", "all created by the agent")}${kpi(os.filter(o => o.status === "OPEN").length, "OPEN", "dispatched automatically")}\n   ${kpi(os.filter(o => o.status === "NEEDS_REVIEW").length, "NEEDS REVIEW", "human exception")}${kpi(by("CRITICAL"), "P1 · CRITICAL")}${kpi(by("HIGH"), "P2 · HIGH")}${kpi(by("MEDIUM"), "P3 · MEDIUM")}</div>\n  <div class="card"><h2>Work-order queue</h2><p class="note">Sorted by priority, then due date. Click an order to see the full card.</p>\n  <div class="filters"><label>Status <select id="oSt"><option value="all">all</option><option ${state.status === "OPEN" ? "selected" : ""}>OPEN</option><option ${state.status === "NEEDS_REVIEW" ? "selected" : ""}>NEEDS_REVIEW</option></select></label>\n  <input type="search" id="oQ" placeholder="search engine, reason…" value="${esc(state.q)}"><span class="muted" id="oCount"></span></div>\n  <div class="scroll"><table><thead><tr><th>Order</th><th>Priority</th><th>Status</th><th class="n">Engine</th><th class="n">Due in</th><th class="n">RUL</th><th>Range</th><th>Why</th></tr></thead><tbody id="oBody"></tbody></table></div>\n  <div id="oCard" style="margin-top:12px"></div></div>`;\n  const body = () => { const q = state.q.toLowerCase();\n    const rows = os.filter(o => (state.status === "all" || o.status === state.status) && (!q || JSON.stringify(o).toLowerCase().includes(q)))\n      .sort((a, b) => a.priority.localeCompare(b.priority) || a.due_within_cycles - b.due_within_cycles);\n    $("oCount").textContent = `${rows.length} orders`;\n    $("oBody").innerHTML = rows.map(o => `<tr class="click" data-id="${esc(o.work_order_id)}"><td style="font-family:ui-monospace,Consolas,monospace;font-size:12px">${esc(o.work_order_id)}</td>\n     <td>${chip(o.severity)} <span class="muted">${o.priority}</span></td><td>${o.status === "OPEN" ? "OPEN" : `<span class="tag warn">NEEDS REVIEW</span>`}</td><td class="n">${o.unit}</td>\n     <td class="n">${o.due_within_cycles === 0 ? "<b>now</b>" : o.due_within_cycles + " cyc"}</td><td class="n">${f0(o.predicted_rul)}</td><td>${rangeBar(o.rul_lower, o.predicted_rul, o.rul_upper)}</td>\n     <td style="max-width:360px">${split(o.review_reasons).map(r => `<span class="tag warn">${esc(r)}</span>`).join("")}${split(o.reasons).slice(0, 2).map(r => `<span class="tag">${esc(r)}</span>`).join("")}</td></tr>`).join(""); };\n  body();\n  $("oSt").onchange = e => { state.status = e.target.value; body(); };\n  $("oQ").oninput = e => { state.q = e.target.value; body(); };\n  $("oBody").onclick = e => { const tr = e.target.closest("tr"); if (!tr) return; $("oCard").innerHTML = woCard(os.find(o => o.work_order_id === tr.dataset.id)); };\n};\n\n// live replay of the validation engines, cycle by cycle\nlet timer = null;\nfunction stopPlay() { if (timer) { clearInterval(timer); timer = null; } }\nVIEWS.replay = () => {\n  const d = D(), R = d.replay_traj, units = Object.keys(R).map(Number).sort((a, b) => a - b);\n  const maxC = Math.max(...units.map(u => R[u].c0 + R[u].pred.length - 1));\n  const stat = Object.fromEntries(d.engine_stats.map(s => [s.unit, s]));\n  let cyc = Math.min(150, maxC), sel = state.unit != null && R[state.unit] ? state.unit : units[0];\n  const fr = d.fleet_replay || {};\n  $("view").innerHTML = `<div class="kpis">${kpi(units.length, "validation engines replayed", "streamed cycle by cycle; cell = predicted RUL")}${kpi(f0(fr.median_rul_at_first_order), "median RUL at first order", "cycles of warning")}\n   ${kpi(f0(fr.median_rul_at_first_high), "median RUL at first HIGH")}${kpi(pct((fr.engines_critical_before_failure_pct ?? 0) / 100), "reached CRITICAL before failure")}\n   ${kpi(f0(fr.min_cycles_left_at_service), "min cycles left at service", "0 would be a failure")}${kpi(fr.engines_with_downgrade_or_closure ?? "–", "orders downgraded or closed", "hysteresis: 5 calmer cycles")}</div>\n  <div class="card" style="margin-bottom:14px"><div class="play"><button id="pBtn">▶ Play</button><b id="pCyc"></b><input type="range" id="pR" min="1" max="${maxC}" value="${cyc}" aria-label="Cycle">\n   <label class="sub">speed <select id="pSp"><option value="120">1×</option><option value="40" selected>3×</option><option value="12">10×</option></select></label></div>\n   <div class="strip" id="pStrip"></div><div class="legend">${SEV.map(s => `<span><i style="background:var(${SEVVAR[s]});height:8px"></i>${s}</span>`).join("")}<span><i style="background:var(--grid);height:8px"></i>serviced by the agent before failure</span><span>outlined = sent to human review</span></div></div>\n  <div class="two"><div class="card"><h2 id="pTitle"></h2><p class="note" id="pNote"></p><div id="pChart"></div>${trajLegend(true)}</div>\n   <div class="card"><h2>Agent event log</h2><p class="note">Order created / escalated / downgraded / closed, up to the current cycle (newest first).</p><div class="feed" id="pFeed"></div></div></div>`;\n  const draw = () => {\n    $("pCyc").textContent = `cycle ${cyc}`; $("pR").value = cyc;\n    $("pStrip").innerHTML = units.map(u => { const t = R[u], st = stat[u], end = st && !st.failed ? st.service_cycle : t.c0 + t.pred.length - 1;\n      const done = cyc >= end, ii = Math.max(0, Math.min(cyc, end) - t.c0), sv = SEV[t.sev[ii]];\n      return `<div class="cell ${done ? "done" : ""} ${!done && t.review[ii] ? "rv" : ""}" data-u="${u}" style="background:var(${SEVVAR[sv]});${sv === "MEDIUM" && !done ? "color:#0b0b0b" : ""}${u === sel ? ";outline:2px solid var(--accent);outline-offset:1px" : ""}"\n       data-tip="engine ${u}${done ? (st && !st.failed ? ` · serviced at cycle ${st.service_cycle}, ${st.cycles_left_at_service} cycles before it would have failed` : ` · failed at cycle ${end}`) : ` · ${sv}<br>predicted RUL ${f0(t.pred[ii])}, actual ${t.true[ii]}`}">${u}<br>${done ? (st && !st.failed ? "serviced" : "FAILED") : f0(t.pred[ii])}</div>`; }).join("");\n    const t = R[sel], st = stat[sel];\n    $("pTitle").innerHTML = `Engine ${sel}`;\n    $("pNote").textContent = st ? `Life ${st.life} cycles · first order at RUL ${st.first_order_rul}, first HIGH at ${st.first_high_rul}, first CRITICAL at ${st.first_critical_rul ?? "–"} · agent service at cycle ${st.service_cycle} (${st.cycles_left_at_service} cycles before failure)` : "";\n    $("pChart").innerHTML = trajChart(t, {cursor: cyc}); bindTraj($("pChart"), t);\n    const ev = d.events.filter(e => e.cycle <= cyc).sort((a, b) => b.cycle - a.cycle || a.unit - b.unit).slice(0, 150);\n    $("pFeed").innerHTML = ev.map(e => `<div><span class="muted">c${e.cycle}</span> · engine <b>${e.unit}</b> · ${e.event} ${chip(e.severity)} <span class="muted">RUL ${f0(e.rul)} (actual ${e.true})</span></div>`).join("") || \'<p class="muted">No orders yet.</p>\';\n  };\n  draw();\n  $("pR").oninput = e => { cyc = +e.target.value; draw(); };\n  $("pStrip").onclick = e => { const c = e.target.closest(".cell"); if (c) { sel = +c.dataset.u; state.unit = sel; draw(); } };\n  $("pBtn").onclick = () => { if (timer) { stopPlay(); $("pBtn").textContent = "▶ Play"; return; }\n    if (cyc >= maxC) cyc = 1; $("pBtn").textContent = "❚❚ Pause";\n    timer = setInterval(() => { cyc++; if (cyc >= maxC) { stopPlay(); $("pBtn").textContent = "▶ Play"; } draw(); }, +$("pSp").value); };\n};\n\nVIEWS.mlops = () => {\n  const d = D(), p = d.promotion || {}, cfg = d.config || {};\n  const psi = Object.entries(d.drift_psi || {}).sort((a, b) => b[1] - a[1]);\n  const maxPsi = psi.length ? psi[0][1] : 0;\n  const test = p.promotion_test || "superiority (older run)";\n  const bootRows = [["P(challenger better)", p.bootstrap_prob_new_better], ["P(worse by > margin)", p.bootstrap_prob_new_worse_by_margin], ["P(NASA clearly worse)", p.bootstrap_prob_nasa_clearly_worse]].filter(r => r[1] != null);\n  const cv = d.cv_results || {};\n  $("view").innerHTML = `<div class="kpis">${kpi("v" + d.production_version, "production model", esc(d.registry_name) + " · alias production")}${kpi("v" + d.champion_version, "champion", "alias champion")}\n   ${kpi(f1((d.production_metrics || {}).test_rmse_capped), "test RMSE (capped at 125)", "true RMSE " + f1((d.production_metrics || {}).test_rmse))}\n   ${kpi(pct((d.production_metrics || {}).test_interval_coverage), "80% range coverage", "on test engines")}\n   ${kpi((+maxPsi).toFixed(2), "max drift (PSI)", maxPsi > cfg.psi_alert ? \'<span class="bad">above \' + cfg.psi_alert + "</span>" : "alert at " + cfg.psi_alert)}\n   ${kpi(p.promoted ? \'<span class="ok">promoted</span>\' : \'<span class="bad">kept v\' + p.champion_version + "</span>", "last retrain decision", esc(test))}</div>\n  <div class="grid g2"><div class="card"><h2>Model versions</h2><p class="note">Every run is tracked (MLflow); rollback = repoint the production alias.</p>\n   <div class="scroll"><table><thead><tr><th>Version</th><th>Model</th><th class="n">Train engines</th><th class="n">Val RMSE</th><th class="n">Test RMSE (capped)</th><th class="n">Test RMSE (true)</th><th class="n">Coverage</th><th>Trained</th></tr></thead><tbody>\n   ${d.history.map(h => `<tr><td><b>v${h.version}</b> ${h.version === d.production_version ? \'<span class="tag">production</span>\' : ""}<div class="muted">${esc(h.label)}</div></td><td>${esc(h.model)}</td><td class="n">${h.train_engines}</td>\n   <td class="n">${f1(h.val_rmse_capped)}</td><td class="n">${f1(h.test_rmse_capped)}</td><td class="n">${f1(h.test_rmse)}</td><td class="n">${pct(h.test_interval_coverage)}</td><td class="muted">${String(h.trained_at || "").slice(0, 16).replace("T", " ")}</td></tr>`).join("")}</tbody></table></div></div>\n  <div class="card"><h2>Retrain &amp; promotion gate</h2><p class="note">Trigger: ≥ ${pct(cfg.retrain_min_new_frac)} new data or PSI > ${cfg.psi_alert}. Gate: ${esc(test)}, ${cfg.promote_min_prob ? "decided at " + pct(cfg.promote_min_prob) + " bootstrap confidence" : ""}.</p>\n   <div>${(p.trigger_reasons || []).map(r => `<span class="tag">${esc(r)}</span>`).join("")}</div>\n   <table style="margin-top:8px"><tbody>\n   <tr><td>Validation RMSE (capped)</td><td class="n">v${p.champion_version} ${f1((p.val_rmse_capped || [])[0])} → v${p.challenger_version} <b>${f1((p.val_rmse_capped || [])[1])}</b></td></tr>\n   <tr><td>NASA score per engine (capped)</td><td class="n">${f1((p.val_nasa_capped_mean || [])[0])} → <b>${f1((p.val_nasa_capped_mean || [])[1])}</b></td></tr>\n   ${p.bootstrap_rmse_diff_ci90 ? `<tr><td>RMSE change, 90% bootstrap CI</td><td class="n">${f1(p.bootstrap_rmse_diff_ci90[0])} to ${f1(p.bootstrap_rmse_diff_ci90[1])}</td></tr>` : ""}\n   ${bootRows.map(r => `<tr><td>${r[0]}</td><td class="n">${pct(r[1], 1)}</td></tr>`).join("")}\n   <tr><td>Decision</td><td class="n">${p.promoted ? \'<span class="ok">promote v\' + p.challenger_version + "</span>" : \'<span class="bad">keep v\' + p.champion_version + "</span>"}</td></tr></tbody></table></div>\n  <div class="card"><h2>Data drift (PSI) per sensor</h2><p class="note">New engines vs training data. Above ${cfg.psi_alert} triggers a retrain.</p><div id="mPsi"></div></div>\n  <div class="card"><h2>Model selection (5-fold CV by engine)</h2><p class="note">Capped RMSE, lower is better. The winner becomes the candidate.</p><div id="mCv"></div></div>\n  <div class="card"><h2>Error by true-RUL band (test)</h2><p class="note">Errors concentrate above the 125-cycle cap, where degradation hasn\'t started. Near failure the model is tight.</p><div id="mBand"></div></div>\n  <div class="card"><h2>Calibration</h2><p class="note">80% range after conformal calibration (CV), and coverage by RUL band on test.</p><div id="mCal"></div></div></div>`;\n  $("mPsi").innerHTML = hbars(psi.slice(0, 10).map(([k, v]) => ({label: k, value: v})), {refLine: cfg.psi_alert, refLabel: "alert " + cfg.psi_alert, fmt: v => (+v).toFixed(2)});\n  $("mCv").innerHTML = hbars(Object.entries(cv).map(([k, v]) => ({label: k, value: v.rmse_capped, color: k === d.champion_model ? "--accent" : "--low", tip: `${k}: capped RMSE ${f1(v.rmse_capped)}, NASA/engine ${f1(v.nasa_capped_mean)}`})));\n  $("mBand").innerHTML = hbars(Object.entries(d.test_error_by_band || {}).map(([k, v]) => ({label: `RUL ${k} (${v.engines})`, value: v.rmse, color: k.startsWith("(1") && +k.split(",")[0].slice(1) >= 125 ? "--low" : "--accent", tip: `${k}: RMSE ${v.rmse}, bias ${v.bias}, ${v.share_of_sq_error_pct}% of squared error, ${v.engines} engines`})));\n  const m = d.production_metrics || {}, ci = d.cv_interval || {};\n  $("mCal").innerHTML = hbars([{label: "CV raw", value: ci.raw_coverage * 100, color: "--low"}, {label: "CV calibrated", value: ci.calibrated_coverage * 100},\n    {label: "test RUL 0–30", value: m.test_coverage_rul_0_30 * 100}, {label: "test RUL 30–60", value: m.test_coverage_rul_30_60 * 100}, {label: "test RUL 60–125", value: m.test_coverage_rul_60_125 * 100}],\n    {max: 100, refLine: 80, refLabel: "target 80%", fmt: v => f0(v) + "%"});\n};\n\nVIEWS.quality = () => {\n  const d = D(), ir = d.ingest_report || {}, fi = d.fault_injection || [];\n  const det = fi.filter(f => f.detected).length;\n  const flags = k => d.fleet.filter(f => String(f.review_reasons || "").includes(k)).length;\n  $("view").innerHTML = `<div class="kpis">${kpi((ir.train || {}).rows ?? "–", "training rows ingested", `${(ir.train || {}).engines} engines`)}${kpi((ir.test || {}).rows ?? "–", "test rows ingested", `${(ir.test || {}).engines} engines`)}\n   ${kpi(d.sensors_kept.length + "/21", "sensors kept", "uninformative ones dropped automatically")}${kpi(d.tickets.length, "sensor tickets (real data)", "CHECK_SENSOR work orders")}\n   ${kpi(`${det}/${fi.length}`, "injected faults caught", "stuck / dead / out of range")}</div>\n  <div class="grid g2"><div class="card"><h2>Ingest checks</h2><p class="note">Run on every batch before anything else.</p><table><thead><tr><th>Check</th><th class="n">Train</th><th class="n">Test</th></tr></thead><tbody>\n   ${["rows", "engines", "missing_cells", "non_numeric_cells", "duplicate_unit_cycle", "cycle_gaps"].map(k => `<tr><td>${k.replace(/_/g, " ")}</td><td class="n">${(ir.train || {})[k] ?? "–"}</td><td class="n">${(ir.test || {})[k] ?? "–"}</td></tr>`).join("")}</tbody></table></div>\n  <div class="card"><h2>Sensors</h2><p class="note">Dropped if constant within an operating condition; the rest are z-scored per condition.</p>\n   <div>${d.sensors_kept.map(s => `<span class="tag">${s}</span>`).join("")}</div><p class="note" style="margin-top:10px">Dropped</p><div>${d.sensors_dropped.map(s => `<span class="tag muted">${s}</span>`).join("") || "none"}</div>\n   <p class="note" style="margin-top:10px">Data-trust flags on the test fleet now: imputed ${flags("imputed")} · unseen condition ${flags("operating condition")} · sensor fault ${flags("sensor fault")}</p></div></div>\n  <div class="card" style="margin-top:14px"><h2>Fault-injection test</h2><p class="note">Faults injected into real test engines. The gate must flag them, open a CHECK_SENSOR ticket and send any order to review instead of trusting the bad data.</p>\n   <div class="scroll"><table><thead><tr><th class="n">Engine</th><th>Injected</th><th class="n">From cycle</th><th>Caught</th><th class="n">Flagged at</th><th>Gate says</th><th>Severity clean → faulty</th><th>Order</th><th>Review reason</th></tr></thead><tbody>\n   ${fi.map(f => `<tr><td class="n">${f.unit}</td><td>${esc(f.injected)}</td><td class="n">${f.from_cycle}</td><td>${f.detected ? \'<span class="ok">yes</span>\' : \'<span class="bad">no</span>\'}</td><td class="n">${f.first_flagged_cycle ?? "–"}</td>\n   <td>${esc(f.gate_says)}</td><td style="white-space:nowrap">${chip(f.severity_clean_data)} → ${chip(f.severity_faulty_data)}</td><td>${esc(f.order_status)}</td><td>${split(f.review_reasons).map(r => `<span class="tag warn">${esc(r)}</span>`).join("")}</td></tr>`).join("")}</tbody></table></div></div>`;\n};\n\nVIEWS.impact = () => {\n  const d = D(), a = d.agent_summary || {}, cm = d.agent_confusion || {}, pc = d.policy_comparison || {};\n  const mx = Math.max(...SEV.flatMap(t => SEV.map(p => (cm[t] || {})[p] || 0)));\n  const cost = (DATA.cost || []).filter(r => r.subset === state.subset);\n  const leads = [...new Set(cost.map(r => r.lead))];\n  $("view").innerHTML = `<div class="kpis">${kpi(`${a.urgent_high_or_critical}/${a.urgent_engines}`, "near-failure engines at HIGH+", `95% lower bound ${pct(a.urgent_high_or_critical_ci95_low)}`)}\n   ${kpi(a.urgent_missed, "near-failure engines missed", "no order at all")}${kpi(a.false_alarms_true_rul_gt_low, "false alarms", "HIGH+ while true RUL > 100")}\n   ${kpi(f0(a.exact_severity_match_pct) + "%", "exact severity match")}${kpi(f0(a.review_rate_pct) + "%", "sent to human review")}${kpi(a.work_orders, "work orders created")}</div>\n  <div class="grid g2"><div class="card"><h2>Agent severity vs true severity (test)</h2><p class="note">Rows: true severity from actual RUL. Columns: what the agent decided. Right of the diagonal = agent acted earlier than needed (safe side); left = later.</p>\n   <table class="heat"><thead><tr><th>true ↓ / agent →</th>${SEV.map(s => `<th style="text-align:center">${s}</th>`).join("")}</tr></thead><tbody>\n   ${SEV.map(t => `<tr><th>${t}</th>${SEV.map(p => { const v = (cm[t] || {})[p] || 0, k = v / (mx || 1);\n     return `<td data-tip="true ${t}, agent ${p}: ${v} engines" style="background:color-mix(in srgb, var(--accent) ${Math.round(k * 85)}%, var(--surface));color:${k > .5 ? "#fff" : "var(--ink)"};${t === p ? "outline:1.5px solid var(--ink2)" : ""}">${v}</td>`; }).join("")}</tr>`).join("")}</tbody></table></div>\n  <div class="card"><h2>Maintenance policy on the replayed fleet</h2><p class="note">Validation engines run to failure. Life used = share of each engine\'s life consumed before service.</p>\n   <table><thead><tr><th>Policy</th><th class="n">Unplanned failures</th><th class="n">Life used</th><th class="n">Cycles wasted / engine</th></tr></thead><tbody>\n   ${Object.entries(pc).map(([k, v]) => `<tr><td>${k.startsWith("predictive") ? "<b>" + esc(k) + "</b>" : esc(k)}</td><td class="n">${f0(v.unplanned_failures)}/${f0(v.engines)}</td><td class="n">${f1(v.life_used_pct)}%</td><td class="n">${f1(v.mean_cycles_wasted)}</td></tr>`).join("")}</tbody></table>\n   <div id="iPol" style="margin-top:8px"></div></div></div>\n  ${cost.length ? `<div class="card" style="margin-top:14px"><h2>Maintenance cost (renewal-reward model)</h2><p class="note">Cost per 1,000 operated cycles; planned service = 1, failure = ratio. Lead time: a service planned fewer cycles than this before failure can\'t be done in time and counts as a failure. Fixed interval T* is the cost-optimal schedule fitted on past history.</p>\n   <div class="filters"><label>Lead time <select id="iLead">${leads.map(l => `<option ${l === 10 ? "selected" : ""}>${l}</option>`).join("")}</select> cycles</label></div><div id="iCost"></div></div>` : ""}`;\n  $("iPol").innerHTML = hbars(Object.entries(pc).map(([k, v]) => ({label: `${k.replace(/ \\(.*\\)/, "").replace("fixed interval", "fixed").replace(" agent", "")} · ${f0(v.unplanned_failures)} fail`, value: v.life_used_pct, color: k.startsWith("predictive") ? "--accent" : "--low",\n    tip: `${k}: ${f1(v.life_used_pct)}% life used, ${v.unplanned_failures} failures`})), {max: 100, fmt: v => f0(v) + "%", w: 520});\n  const costTable = () => { const lead = +$("iLead").value, rs = cost.filter(r => r.lead === lead), ratios = [...new Set(rs.map(r => r.ratio))];\n    const pol = [...new Set(rs.map(r => r.policy))];\n    $("iCost").innerHTML = `<div class="scroll"><table><thead><tr><th>Policy</th>${ratios.map(r => `<th class="n">failure = ${r}× service</th>`).join("")}</tr></thead><tbody>\n     ${pol.map(p => `<tr><td>${p.startsWith("agent") ? "<b>predictive agent</b>" : esc(p)}</td>${ratios.map(r => { const x = rs.find(z => z.policy === p && z.ratio === r);\n       const fixed = rs.find(z => z.policy.startsWith("optimal") && z.ratio === r), save = p.startsWith("agent") && fixed ? 1 - x.cost_per_1000_cycles / fixed.cost_per_1000_cycles : null;\n       return `<td class="n">${x.cost_per_1000_cycles.toFixed(2)} <span class="muted">(${x.unplanned_failures} fail${p.startsWith("optimal") ? ", T=" + x.t_star : ""})</span>${save != null ? `<div class="${save > 0 ? "ok" : "bad"}">${save > 0 ? "−" : "+"}${Math.abs(save * 100).toFixed(0)}% vs T*</div>` : ""}</td>`; }).join("")}</tr>`).join("")}</tbody></table></div>`; };\n  if (cost.length) { costTable(); $("iLead").onchange = costTable; }\n};\n\nrender();\n</script></body></html>\n"""\n\nif __name__ == "__main__":\n    root = Path(__file__).resolve().parents[1]\n    run = Path(sys.argv[1]) if len(sys.argv) > 1 else root / "output" / "all_run_v3"\n    out = build(run)\n    print("written:", out, f"({out.stat().st_size / 1e6:.1f} MB)")\n'
print(len(PIPELINE_CELLS), 'pipeline cells loaded')

In [ ]:
import gc, shutil, time, traceback
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
# Seed 42 is the primary run (outputs in artifacts/<SUBSET>, as before). Each extra seed changes the engine splits,
# CV folds and model randomness; its outputs go to artifacts/<SUBSET>_seed<N>. Use [42] for a quick single run.
SEEDS = [42, 1, 2, 3, 4]
CLEAN_START = True

WORK = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path.cwd() / "outputs"
if CLEAN_START:
    for d in ["artifacts", "tracking"] + [f"tracking_seed{s}" for s in SEEDS]:
        shutil.rmtree(WORK / d, ignore_errors=True)

results = {}  # (subset, seed) -> result
for seed in SEEDS:
    tag = "" if seed == 42 else f"_seed{seed}"
    for subset in SUBSETS:
        print(f"\n{'=' * 30} {subset} · seed {seed} {'=' * 30}")
        ns = {"__name__": "__main__", "display": display, "SKIP_ZIP": True}
        t0 = time.time()
        try:
            for i, src in enumerate(PIPELINE_CELLS):
                code = (src.replace('SUBSET = "FD001"', f'SUBSET = "{subset}"')
                           .replace("SEED = 42\n", f"SEED = {seed}\n")
                           .replace('RUN_TAG = ""', f'RUN_TAG = "{tag}"'))
                exec(compile(code, f"<{subset} seed {seed} cell {i}>", "exec"), ns)
            results[(subset, seed)] = {"status": "ok", "minutes": round((time.time() - t0) / 60, 1),
                                       "monitoring": ns["monitoring"]}
        except Exception:
            traceback.print_exc()
            results[(subset, seed)] = {"status": "failed", "minutes": round((time.time() - t0) / 60, 1),
                                       "error": traceback.format_exc(limit=5)}
        finally:
            try:
                import mlflow
                mlflow.end_run()
            except Exception:
                pass
            plt.close("all")
            del ns
            gc.collect()
        print(f"{subset} seed {seed}: {results[(subset, seed)]['status']} in {results[(subset, seed)]['minutes']} min")

## Cross-subset summary

`test_rmse_capped` is the error against min(true RUL, 125), the part the model can reduce; `test_rmse` is against the true RUL. FD002 / FD004 have ~25% of test engines above the cap, which drives their true RMSE and NASA score (see each subset's `test_error_by_band` in `monitoring.json`).

The first table is the primary run (seed 42). The robustness table repeats the key numbers across all seeds: mean, standard deviation, and the worst seed, so a result that only holds for one lucky split shows up here.

In [ ]:
def engines_above_cap(m):
    cap = m["config"]["rul_cap"]
    return int(sum(v["engines"] for k, v in m["test_error_by_band"].items() if float(k.strip("([").split(",")[0]) >= cap))


def summary_row(r):
    if r["status"] != "ok":
        return {"status": "FAILED", "minutes": r["minutes"]}
    m = r["monitoring"]
    p, a = m["production_metrics"], m["agent_summary"]
    return {
        "status": "ok", "model": m["champion_model"], "production_version": m["production_version"],
        "cv_rmse_capped": round(m["cv_results"][m["champion_model"]]["rmse_capped"], 2),
        "test_rmse_capped": round(p["test_rmse_capped"], 2), "test_rmse": round(p["test_rmse"], 2),
        "test_nasa_score": round(p["test_nasa_score"], 1), "test_interval_coverage": round(p["test_interval_coverage"], 3),
        "test_engines": a["engines"], "test_engines_above_cap": engines_above_cap(m),
        "urgent_high_or_critical": f"{a['urgent_high_or_critical']}/{a['urgent_engines']}",
        "urgent_caught": f"{a['urgent_caught']}/{a['urgent_engines']}", "urgent_missed": a["urgent_missed"],
        "exact_severity_pct": round(a["exact_severity_match_pct"], 1), "review_pct": round(a["review_rate_pct"], 1),
        "false_alarms": a["false_alarms_true_rul_gt_low"], "work_orders": a["work_orders"],
        "promoted_v2": m["promotion"].get("promoted"),
        "v2_better_in_bootstrap_pct": round(100 * m["promotion"].get("bootstrap_prob_new_better", float("nan")), 1),
        "val_engines": m["fleet_replay"]["engines"],
        "ours_unplanned_failures": m["policy_comparison"]["predictive agent (ours)"]["unplanned_failures"],
        "ours_life_used_pct": round(m["policy_comparison"]["predictive agent (ours)"]["life_used_pct"], 1),
        "fixed_safe_life_used_pct": round(next(v["life_used_pct"] for k, v in m["policy_comparison"].items()
                                               if "0% failures" in k), 1),
        "median_rul_at_first_order": m["fleet_replay"]["median_rul_at_first_order"],
        "fault_injections_detected": f"{sum(f['detected'] for f in m['fault_injection'])}/{len(m['fault_injection'])}",
        "minutes": r["minutes"],
    }


primary = SEEDS[0]
summary = pd.DataFrame({s: summary_row(results[(s, primary)]) for s in SUBSETS if (s, primary) in results})
print(f"primary run (seed {primary})")
display(summary)

for (subset, seed), r in results.items():
    if r["status"] != "ok":
        print(f"\n{subset} seed {seed} failed:\n{r['error']}")

# robustness across seeds: numeric headline metrics per (subset, seed), then mean / std / worst per subset
def seed_metrics(r):
    m = r["monitoring"]
    p, a, pol = m["production_metrics"], m["agent_summary"], m["policy_comparison"]
    ours = pol["predictive agent (ours)"]
    safe = next(v for k, v in pol.items() if "0% failures" in k)
    return {"test_rmse_capped": p["test_rmse_capped"], "test_rmse": p["test_rmse"],
            "test_nasa_score": p["test_nasa_score"], "test_interval_coverage": p["test_interval_coverage"],
            "urgent_high_or_critical_pct": 100 * a["urgent_high_or_critical"] / a["urgent_engines"],
            "urgent_missed": a["urgent_missed"], "false_alarms": a["false_alarms_true_rul_gt_low"],
            "review_pct": a["review_rate_pct"], "promoted_v2": float(bool(m["promotion"].get("promoted"))),
            "ours_unplanned_failures": ours["unplanned_failures"], "ours_life_used_pct": ours["life_used_pct"],
            "fixed_safe_life_used_pct": safe["life_used_pct"]}


per_seed = pd.DataFrame([{"subset": s, "seed": sd, **seed_metrics(r)} for (s, sd), r in results.items()
                         if r["status"] == "ok"])
robust = None
if len(per_seed) and per_seed.seed.nunique() > 1:
    worst = {"test_rmse_capped": "max", "test_rmse": "max", "test_nasa_score": "max", "test_interval_coverage": "min",
             "urgent_high_or_critical_pct": "min", "urgent_missed": "max", "false_alarms": "max", "review_pct": "max",
             "promoted_v2": "min", "ours_unplanned_failures": "max", "ours_life_used_pct": "min",
             "fixed_safe_life_used_pct": "max"}
    g = per_seed.groupby("subset")
    robust = pd.concat({"mean": g[list(worst)].mean(), "std": g[list(worst)].std(),
                        "worst": g.agg(worst)}, axis=1).T.round(2)
    print(f"robustness across {per_seed.seed.nunique()} seeds {sorted(per_seed.seed.unique())}")
    display(robust)

(WORK / "artifacts").mkdir(parents=True, exist_ok=True)
summary.to_csv(WORK / "artifacts" / "summary_all_subsets.csv")
(WORK / "artifacts" / "summary_all_subsets.json").write_text(summary.to_json(indent=2))
per_seed.to_csv(WORK / "artifacts" / "summary_per_seed.csv", index=False)
if robust is not None:
    robust.to_csv(WORK / "artifacts" / "summary_robustness.csv")

bundle_dir = WORK / "all_subsets_outputs"
shutil.rmtree(bundle_dir, ignore_errors=True)
# extra seeds are for robustness numbers only: keep their results, drop their model files to keep the zip small
skip_models = lambda d, names: [n for n in names if "_seed" in str(d) and (n.endswith(".joblib") or n == "models")]
shutil.copytree(WORK / "artifacts", bundle_dir / "artifacts", ignore=skip_models)
if (WORK / "tracking").exists():
    shutil.copytree(WORK / "tracking", bundle_dir / "tracking")  # primary seed's MLflow store only
# monitoring dashboard: one offline HTML page built from the primary seed's artifacts
dash_ns = {"__name__": "build_dashboard"}
exec(compile(DASHBOARD_SRC, "build_dashboard.py", "exec"), dash_ns)
print("dashboard:", dash_ns["build"](WORK, bundle_dir / "dashboard.html"))
zip_path = shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir)
shutil.rmtree(bundle_dir)
print("outputs:", zip_path)